# Linear independence of 1, $\zeta(2)$, $L(2,\chi_{-3})$ from one CMF trajectory

**22 October 2026.** Supplement to *New Directions in Irrationality*, [arXiv:2610.27799](https://arxiv.org/abs/2610.27799): an explicit Apéry-style proof of the theorem of Calegari, Dimitrov and Tang, [arXiv:2408.15403](https://arxiv.org/abs/2408.15403).

Let $C=L(2,\chi_{-3})$, $Z=\zeta(2)$, and
\[
\alpha_{A,B}=A(C+2Z/3)+B(C-2Z/3).
\]
For every fixed nonzero integer pair $(A,B)$, the argument below gives
\[
\liminf_{N\to\infty}\delta_N(A,B)>0.00010.
\]
The finer certified lower comparison is approximately **0.0001033289383**.
This implies irrationality of every such mixture, and hence linear independence
of $1,C,Z$.

The analytic section evaluates the initial MG state exactly, writes it as a
positive four-variable beta integral, and proves that it is the recessive
solution. Matrix asymptotics and the terminal-mode check then identify both
dual limits and their exponential error bound.

**No limit or error-rate assumption is retained.** The arithmetic interpretation
uses the written valuation lemmas in the appendix. The finite rational
calculations and final interval comparisons are checked by Python; this is a
computer-assisted argument, not a proof-assistant formalization.

Run all cells in order. The notebook embeds its endpoint evaluator, analytic
checks and arithmetic verifier. It reconstructs both matrices from RamanujanTools. Dependencies:
`ramanujantools`, `sympy`, `python-flint`; Jupyter is needed only as a notebook frontend.
Fresh construction and certificate evaluation take several minutes. All generated
files go into `notebook_outputs/` beside the running notebook.


In [1]:
import sys, json, math, time, hashlib, subprocess, importlib.metadata, csv
from pathlib import Path
import sympy as sp
from ramanujantools import Matrix, Position
from ramanujantools.cmf import MeijerG, pFq
from ramanujantools.asymptotics import SeriesMatrix, Reducer
from flint import fmpq as FQ, fmpq_mat as FMat, arb, ctx
from IPython.display import display, Markdown

sys.set_int_max_str_digits(0)
OUT = Path.cwd() / 'notebook_outputs'
OUT.mkdir(exist_ok=True)
n, x = sp.symbols('n x')
REBUILD_CMFS = False  # Set True to ignore the cache; a clean directory always reconstructs.
FRESH_ARITHMETIC = False  # Reuse the source-hashed 1,158-chamber checkpoint; True recomputes it.
versions = {p: importlib.metadata.version(p)
            for p in ('ramanujantools', 'sympy', 'python-flint')}
print(versions)
print('Output directory:', OUT)


{'ramanujantools': '0.0.8', 'sympy': '1.13.3', 'python-flint': '0.8.0'}
Output directory: notebook_outputs


## 1. The two CMFs and their trajectories

Both use $z=1$. The product convention is $W_N=Q(0)\cdots Q(N-1)$,
$U_N=W_N^{-T}=Q(0)^{-T}\cdots Q(N-1)^{-T}$.
The inverse-transpose product has the **same chronological order**.

The Meijer-G constructor has a four-dimensional matrix here. Its preserved
three-dimensional block is the existing restriction used throughout this notebook.
The forward CMF also has native dimension four and has the moving rank-three image
of $K(n)$. An invertible duality or gauge alone does not reduce dimension.


In [2]:
mg_start = (-1,-1,2,sp.Rational(7,3),sp.Rational(5,2),
            0,0,0,sp.Rational(1,3),sp.Rational(1,2))
mg_direction = (-10,-8,12,6,5, 0,1,2,0,2)
forward_start = mg_start[:5] + (1,1,sp.Rational(4,3),sp.Rational(3,2))
forward_direction = mg_direction[:5] + (1,2,0,2)
mg_axes = sp.symbols('a0:5') + sp.symbols('b0:5')
forward_axes = sp.symbols('x0:5') + sp.symbols('y0:4')
specifications = {
    'MG(5,2,5,5,1)': (mg_start, mg_direction),
    'pFq(5,4,1)': (forward_start, forward_direction),
}
for family, (start, direction) in specifications.items():
    print(family, '\n  initial =', start, '\n  direction =', direction)

def build_cmf(family, axes, start, direction):
    return family.trajectory_matrix(Position(dict(zip(axes,direction))),
                                    Position(dict(zip(axes,start))))

cached = OUT / 'exact_matrices.json'
begun = time.monotonic()
if REBUILD_CMFS or not cached.exists():
    M = build_cmf(pFq(5,4,1), forward_axes, forward_start, forward_direction)
    print('Forward matrix constructed.', flush=True)
    G = build_cmf(MeijerG(5,2,5,5,1), mg_axes, mg_start, mg_direction)
    data = {key: [[str(z) for z in row] for row in value.tolist()]
            for key,value in [('MG55',G),('PFQ54',M)]}
    cached.write_text(json.dumps(data, indent=2)+'\n')
else:
    data = json.loads(cached.read_text())
    G, M = (Matrix([[sp.sympify(z,locals={'n':n}) for z in row]
                    for row in data[key]]) for key in ('MG55','PFQ54'))
print('Native dimensions:', G.shape, M.shape)
print('Construction seconds:', round(time.monotonic()-begun,3))
J = Matrix.vstack(Matrix.eye(3), Matrix.zeros(1,3))
Q = G[:3,:3]
assert (G*J-J*Q).applyfunc(sp.cancel) == sp.zeros(4,3)
assert Q.subs({n:0}).det() != 0  # All-n gamma determinant identity is checked in Section 7.
print('Exact preserved block: G(n)J = JQ(n); restricted dimension = 3.')


MG(5,2,5,5,1) 
  initial = (-1, -1, 2, 7/3, 5/2, 0, 0, 0, 1/3, 1/2) 
  direction = (-10, -8, 12, 6, 5, 0, 1, 2, 0, 2)
pFq(5,4,1) 
  initial = (-1, -1, 2, 7/3, 5/2, 1, 1, 4/3, 3/2) 
  direction = (-10, -8, 12, 6, 5, 1, 2, 0, 2)
Native dimensions: (4, 4) (4, 4)
Construction seconds: 0.234
Exact preserved block: G(n)J = JQ(n); restricted dimension = 3.


## 2. Fixed readouts and the initial matrix depending on $A,B$

With $v_N=U_N e_0$, set $r_C=Xv_N/Dv_N$, $r_Z=Yv_N/Dv_N$.
For a fixed mixture the complete two-row initial/readout matrix is
\[
I_{A,B}=\begin{pmatrix}3(A+B)X+2(A-B)Y\\3D\end{pmatrix},\qquad
r_{A,B,N}=\frac{(I_{A,B}v_N)_0}{(I_{A,B}v_N)_1}.
\]
These coefficients are fixed independently of depth. Section 6A proves
$r_C\to C$ and $r_Z\to Z$ from the initial MG state and its recessiveness.


In [3]:
D = Matrix([[243648,-498096,264600]])
X = Matrix([[190344,-389162,206743]])
Y = Matrix([[400824,-819330,435234]])
e0 = Matrix([1,0,0])
A, B = sp.symbols('A B', integer=True)
I_AB = Matrix.vstack(3*(A+B)*X + 2*(A-B)*Y, 3*D)
assert Matrix.vstack(D,X,Y).det() == -117573120
display(I_AB)


Matrix([
[1372680*A - 230616*B, -2806146*A + 471174*B, 1490697*A - 250239*B],
[              730944,              -1494288,               793800]])

## 3. Exact forward/dual pairing

The inverse-free identity checked below is
\[
M(n)K(n+1)Q(n)^T=c(n)K(n),
\]
equivalently $M(n)K(n+1)=c(n)K(n)Q(n)^{-T}$.
Let $L K(0)=I_3$. Then the forward readout is $I_{A,B}L$ applied to
\[
M(0)\cdots M(N-1)K(N)e_0.
\]
Its terminal column is **$K(N)e_0$** and depends on $N$.
The common scalar $\prod c(j)$ cancels from the rational quotient.


In [4]:
K = Matrix([
 [(19440*n**4+13812*n**3+3219*n**2+308*n+11)/3,
  (6*n+1)*(720*n**2+443*n+46)/6,-5*(216*n**2+75*n+7)/6],
 [-(4320*n**3+4410*n**2+1045*n+69)/6,(360*n**2+115*n+8)/2,1],
 [-(1080*n**2+315*n+7)/6,-2,0], [3,0,0]])
ar = (sp.rf(12*n+2,12)*sp.rf(6*n+sp.Rational(7,3),6)*sp.rf(5*n+sp.Rational(5,2),5)
      /(sp.rf(2*n+1,2)*sp.rf(2*n+sp.Rational(3,2),2)*sp.rf(10*n+2,10)
        *sp.rf(8*n+2,8)*(n+1)))
c = sp.factor(1/ar)
begun = time.monotonic()
residual = (M*K.subs({n:n+1})*Q.T-c*K).applyfunc(sp.cancel)
assert residual == sp.zeros(4,3)
assert K[1:4,:].det() == 6
L = Matrix([[0,0,0,sp.Rational(1,3)],
            [0,0,sp.Rational(-1,2),sp.Rational(-7,36)],
            [0,1,2,sp.Rational(83,18)]])
assert L*K.subs({n:0}) == sp.eye(3)
print('All 12 pairing residuals are exactly zero.')
print('K has rank 3 for every n; lower 3-by-3 minor = 6.')
print('Pairing verification seconds:', round(time.monotonic()-begun,3))
display(K)
display(sp.Eq(sp.Symbol('c(n)'), c))
print('Forward A,B-dependent initial/readout matrix:')
display(I_AB*L)


All 12 pairing residuals are exactly zero.
K has rank 3 for every n; lower 3-by-3 minor = 6.
Pairing verification seconds: 22.357
Forward A,B-dependent initial/readout matrix:


Matrix([
[6480*n**4 + 4604*n**3 + 1073*n**2 + 308*n/3 + 11/3, (6*n + 1)*(720*n**2 + 443*n + 46)/6, -180*n**2 - 125*n/2 - 35/6],
[            -720*n**3 - 735*n**2 - 1045*n/6 - 23/2,              180*n**2 + 115*n/2 + 4,                          1],
[                         -180*n**2 - 105*n/2 - 7/6,                                  -2,                          0],
[                                                 3,                                   0,                          0]])

Eq(c(n), 720*(n + 1)**3*(2*n + 1)*(4*n + 3)*(4*n + 5)*(5*n + 1)*(5*n + 2)*(5*n + 3)*(5*n + 4)*(8*n + 3)*(8*n + 5)*(8*n + 7)*(8*n + 9)*(10*n + 3)/((3*n + 1)*(3*n + 2)*(6*n + 1)*(6*n + 5)*(9*n + 5)*(9*n + 8)*(9*n + 11)*(10*n + 13)*(12*n + 5)*(12*n + 7)*(12*n + 11)*(12*n + 13)*(18*n + 7)*(18*n + 13)*(18*n + 19)))

Matrix([
[0, 1490697*A - 250239*B, 4384467*A - 736065*B, 7876969*A - 1322369*B],
[0,               793800,              2334744,               4194504]])

## 4. Exact endpoint approximants

The following evaluator is independent of matrix multiplication. It sums exact
residues and harmonic moments of
\[
F_N(s)=\frac{(s+1)_{10N+1}(s+1-N)_{9N+1}}
{(2N-s)_{10N+2}(1/3-s)_{6N+2}(2N+1/2-s)_{3N+2}}.
\]
For its poles $\rho=j+r$, write $v_\rho=-\operatorname{Res}_{s=\rho}F_N(s)$.
The four rows are $B_0=-\sum v_\rho h_r(j)(1,\rho,\rho^2,\rho^3)$
and the integer/third/half residue moment rows. The exact residue constraint is
$B_I+B_T+B_H=0$ in columns 0,1,2. Let
$\widehat B_N=(B_0,B_T,B_H)[:,0:3]$, and take the cross product of its columns 1,2:
\[
w_N=\widehat B_N[:,1]\times\widehat B_N[:,2],\quad
r_C=\frac{4w_1-6w_2}{18w_0},\quad r_Z=\frac{w_2}{2w_0},\quad
r_{A,B}=\frac{2((A+B)w_1-3Bw_2)}{9w_0}.
\]
The appendix gives the transport argument for all $N$. The next checks additionally
compare exact endpoint fractions with both matrix products at small unused depths.


In [5]:
from flint import fmpq as Rat, fmpz as ZZ
def product(values):
    values=list(values)
    while len(values)>1:
        values=[values[i]*values[i+1] if i+1<len(values) else values[i]
                for i in range(0,len(values),2)]
    return values[0] if values else ZZ(1)

def qproduct(values):
    values=list(values)
    return Rat(product(ZZ(x.numerator) for x in values),
             product(ZZ(x.denominator) for x in values))

def summation(a,L,n,classes,lo,hi):
    if hi-lo==1:
        s=a+lo
        P=Rat(1);H=Rat(0)
        if lo+1<L:
            P=(s+10*n+2)*(s+8*n+2)/((s+1)*(s+1-n))
            for b,K,_ in classes: P *= (s+1-b-K)/(s+1-b)
            H=1/(s+1)**2
        return P,H,[s**k for k in range(4)],[Rat(0)]*4
    mid=(lo+hi)//2
    P,H,U,V=summation(a,L,n,classes,lo,mid)
    p,h,u,v=summation(a,L,n,classes,mid,hi)
    return P*p,H+h,[U[k]+P*u[k] for k in range(4)], [V[k]+P*(v[k]+H*u[k]) for k in range(4)]

def harmonic(offset,lo,hi):
    if hi-lo<=16: return sum((1/(offset+j)**2 for j in range(lo,hi)),Rat(0))
    mid=(lo+hi)//2
    return harmonic(offset,lo,mid)+harmonic(offset,mid,hi)

def endpoint(n):
    classes=[(Rat(2*n),10*n+2,0),(Rat(1,3),6*n+2,1),(Rat(2*n)+Rat(1,2),3*n+2,2)]
    B=[[Rat(0) for k in range(4)] for i in range(4)]
    for a,L,cl in classes:
        numerator=qproduct(a+1+j for j in range(10*n+1))*qproduct(a+1-n+j for j in range(9*n+1))
        denominator=qproduct(b+j-a for b,K,other in classes for j in range(K) if not(other==cl and j==0))
        v=numerator/denominator
        j0=2*n if cl in (0,2) else 0
        h=harmonic((Rat(0),Rat(1,3),Rat(1,2))[cl],1 if cl==0 else 0,j0+1)
        _,_,U,V=summation(a,L,n,classes,0,L)
        for k in range(4):
            B[cl+1][k]=v*U[k]
            B[0][k]-=v*(V[k]+h*U[k])
    return B

def cofactors(B):
    u=[B[i][1] for i in (0,2,3)]
    v=[B[i][2] for i in (0,2,3)]
    return [u[1]*v[2]-u[2]*v[1],u[2]*v[0]-u[0]*v[2],u[0]*v[1]-u[1]*v[0]]

def qcontent(values):
    return Rat(math.gcd(*(int(v.numerator) for v in values)),
             math.lcm(*(int(v.denominator) for v in values)))


In [6]:
def smat(B):
    return Matrix([[sp.Rational(str(z)) for z in row] for row in B])
def bhat(N):
    BB = endpoint(N)
    return smat([[BB[i][j] for j in range(3)] for i in (0,2,3)])
def ratios(w):
    return ((4*w[1]-6*w[2])/(18*w[0]), w[2]/(2*w[0]))
def mix(w,a,b):
    return 2*((a+b)*w[1]-3*b*w[2])/(9*w[0])

B0 = bhat(0)
assert B0.det() == -sp.Rational(1944,35)
Ai = B0.inv().T
assert 3888*Ai[0,:] == D
assert 3888*(4*Ai[1,:]-6*Ai[2,:])/18 == X
assert 3888*Ai[2,:]/2 == Y
U = Matrix.eye(3); Wf = Matrix.eye(4); cc = sp.S.One
for N in range(4):
    if N:
        j=N-1
        Qj=Q.subs({n:j}); Mj=M.subs({n:j})
        assert bhat(N) == -bhat(j)*Qj
        U=U*Qj.inv().T; Wf=Wf*Mj; cc*=c.subs(n,j)
    v=U*e0
    vf=Wf*K.subs({n:N})*e0
    assert vf == cc*K.subs({n:0})*v
    rc,rz=ratios(cofactors(endpoint(N)))
    assert sp.Rational(str(rc)) == (X*v)[0]/(D*v)[0]
    assert sp.Rational(str(rz)) == (Y*v)[0]/(D*v)[0]
    for a,b in [(1,0),(0,1),(1,1),(2,-3)]:
        row=I_AB.subs({A:a,B:b})
        assert sp.Rational(str(mix(cofactors(endpoint(N)),a,b))) == (row*L*vf)[0]/(row*L*vf)[1]
print('Endpoint, dual and moving-terminal forward fractions agree exactly at N=0,1,2,3.')
display(B0)


Endpoint, dual and moving-terminal forward fractions agree exactly at N=0,1,2,3.


Matrix([
[-789349/1680, -108221/140, -35787/35],
[     1647/35,     3636/35,   5328/35],
[      768/35,    -1116/35,  -2808/35]])

## 5. Numerical approximations with reduced scalar denominators

The targets are evaluated independently using Arb's Hurwitz zeta:
$C=(\zeta(2,1/3)-\zeta(2,2/3))/9$. All rational fractions are reduced
before extracting $q_N$. The CSV contains the complete integers $p_N,q_N$,
error enclosures, precision, runtime and endpoint matrix height.

These finite-depth values illustrate the construction. They are not inputs to
the arithmetic certificate or evidence by themselves for a positive asymptotic delta.


In [7]:
depths = [20,40,80,160,320,640]
cases = [('C',None),('zeta(2)',None),('plus',(1,0)),('minus',(0,1)),
         ('2C',(1,1)),('4zeta(2)/3',(1,-1)),('A=2 B=1',(2,1)),
         ('A=1 B=2',(1,2)),('A=3 B=-1',(3,-1)),('A=-2 B=3',(-2,3))]
records=[]; moments={}
for N in depths:
    begun=time.monotonic(); ctx.prec=100*N+512
    BB=endpoint(N); w=cofactors(BB); moments[N]=w
    Cvalue=(arb(2).zeta(arb(1)/3)-arb(2).zeta(arb(2)/3))/9
    Zvalue=arb(2).zeta()
    rc,rz=ratios(w)
    height=max(max(abs(int(z.numerator)).bit_length(),int(z.denominator).bit_length())
               for row in BB for z in row)
    for name,ab in cases:
        if name=='C': rational,target=rc,Cvalue
        elif name=='zeta(2)': rational,target=rz,Zvalue
        else:
            a,b=ab; rational=mix(w,a,b)
            target=(a+b)*Cvalue+arb(2)*(a-b)*Zvalue/3
        error=abs(target-arb(rational)); assert error>0
        h=arb(rational.denominator).log(); ee=-error.log()
        delta=ee/h-1
        records.append(dict(N=N,target=name,A=None if ab is None else ab[0],
            B=None if ab is None else ab[1],p=str(rational.numerator),q=str(rational.denominator),
            absolute_error=error.str(20),delta=delta.str(20),delta_mid=float(delta.mid()),
            log_q=float(h.mid()),minus_log_error=float(ee.mid()),
            digits_correct=float((ee/arb(10).log()).mid()),precision_bits=ctx.prec,
            matrix_entry_height_bits=height,runtime_seconds=time.monotonic()-begun))
    print('N=%d complete in %.2fs; precision %d bits' % (N,time.monotonic()-begun,ctx.prec),flush=True)
with (OUT/'approximants.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
print('\nReduced-denominator delta (finite depth):')
print('target'.ljust(16)+''.join(f'{N:>12}' for N in depths))
for name,_ in cases:
    rr=[r for r in records if r['target']==name]
    print(name.ljust(16)+''.join(f'{r["delta_mid"]:12.8f}' for r in rr))

# Independent higher-precision retest, with coefficients already frozen.
ctx.prec=100*80+1024
Ccheck=(arb(2).zeta(arb(1)/3)-arb(2).zeta(arb(2)/3))/9
rc,_=ratios(moments[80]); deltacheck=-1-abs(Ccheck-arb(rc)).log()/arb(rc.denominator).log()
assert abs(float(deltacheck.mid())-next(r['delta_mid'] for r in records if r['N']==80 and r['target']=='C'))<1e-13
print('N=80 higher-precision check passed.')
print('\nLater-window secant slopes (same two depths for E and H):')
slopes=[]
for name in ['C','plus','minus']:
    rr={r['N']:r for r in records if r['target']==name}
    for a,b in [(80,160),(160,320),(320,640)]:
        es=(rr[b]['minus_log_error']-rr[a]['minus_log_error'])/(b-a)
        hs=(rr[b]['log_q']-rr[a]['log_q'])/(b-a)
        slopes.append(dict(target=name,start=a,end=b,error_slope=es,height_slope=hs,delta_slope=es/hs-1))
        print(f'{name:6} {a:3}->{b:3}: E slope={es:.9f}, H slope={hs:.9f}, delta slope={es/hs-1:+.9f}')


N=20 complete in 0.01s; precision 2512 bits
N=40 complete in 0.02s; precision 4512 bits
N=80 complete in 0.08s; precision 8512 bits
N=160 complete in 0.33s; precision 16512 bits
N=320 complete in 1.58s; precision 32512 bits
N=640 complete in 7.69s; precision 64512 bits

Reduced-denominator delta (finite depth):
target                    20          40          80         160         320         640
C                 0.01181338  0.01556407  0.00810235  0.00604332  0.00041314  0.00013945
zeta(2)           0.02235846  0.00735725  0.00898742  0.00610254  0.00042283 -0.00026701
plus              0.01386030  0.01219753  0.00860519  0.00636346  0.00043306 -0.00032167
minus             0.06203072  0.05634088  0.05403514  0.05421902  0.04893296  0.04922833
2C                0.01181975  0.01556831  0.00810345  0.00604373  0.00041316  0.00013946
4zeta(2)/3        0.02236352  0.00735807  0.00898793  0.00610271  0.00042284 -0.00026702
A=2 B=1           0.01380423  0.00919462  0.00772752  0.00570495

## 6. Matrix asymptotics using RamanujanTools

Use $D_g(n)=\operatorname{diag}(1,n^{-1},n^{-2})$, $s(n)=Q_{00}(n)$,
\[
\mathcal A(n)=D_g(n)^{-1}Q(n)D_g(n+1)/s(n)=T+T_1/n+O(n^{-2}).
\]
The series is obtained from the **actual rational matrix**, not from fitted data.
For the transposed column system, the exact rational first-order gauge below gives
\[
T_1^T+T^T H-H T^T=bT^T,\qquad
b=2042824873127/3671732351280.
\]
Thus a gauge $I+H/n$ leaves $(1+b/n)T^T+O(n^{-2})$.
Diagonalize the constant matrix over its cubic splitting field. All first-order
scalar blocks are \(\lambda_i(1+b/n)+O(n^{-2})\).

The next cell calls the installed **RamanujanTools `Reducer.asymptotic_growth`**
on these exact scalar blocks, with algebraic `CRootOf` values. This exact
preconditioning avoids an expensive generic primitive-element computation on
the whole cubic splitting field. No floating matrix or rounded eigenvalue is an input.


In [8]:
normalized = Matrix(3,3,lambda i,j:sp.cancel(n**i*Q[i,j]/Q[0,0]/(n+1)**j))
series = SeriesMatrix.from_matrix(normalized.T,var=n,p=1,precision=3)
Tc, T1c = (Matrix(z) for z in series.coeffs[:2]); T=Tc.T
bpower=sp.Rational(2042824873127,3671732351280)
H=Matrix([[sp.Rational(107,660),-sp.Rational(13,396),-sp.Rational(59,2640)],
          [sp.Rational(323,198),-sp.Rational(2483,7920),-sp.Rational(101,1760)],
          [0,0,0]])
assert T1c+Tc*H-H*Tc == bpower*Tc
poly=sp.Poly(sp.Matrix(T).charpoly(x).as_expr(),x)
expected=[1,-sp.Rational(162060225797,305977695940),
          sp.Rational(40100849267287098349209,374489401651004369934400),
          -sp.Rational(25,7161587766945221818712445059146)]
assert poly.all_coeffs()==expected
assert sp.discriminant(poly.as_expr(),x)<0
print('Limiting rational matrix T:'); display(T)
print('Characteristic polynomial:');display(poly.as_expr())
print('Exact first-order gauge H:');display(H)
print('First-order gauge residual: zero in all 9 entries.')

roots=[sp.CRootOf(poly.as_expr(),i) for i in range(3)]
growths=[]
for root in roots:
    scalar=SeriesMatrix([Matrix([[root]]),Matrix([[bpower*root]]),Matrix([[0]])])
    growth=Reducer(scalar,factorial_power=0,precision=3,p=1).asymptotic_growth()[0]
    assert growth.exp_base==root
    assert sp.simplify(growth.polynomial_degree-bpower)==0
    assert growth.factorial_power==0 and growth.sub_exp==0 and growth.log_power==0
    growths.append(growth)
    print('lambda =',sp.N(root,40),'; power of n =',sp.N(growth.polynomial_degree,18))
r=roots[0]
assert r.is_real and r>0
R=sp.sqrt(-expected[-1]/r)
E=sp.log(R/r)
print('r =',sp.N(r,55))
print('R =',sp.N(R,55))
print('E = log(R/r) =',sp.N(E,55))


Limiting rational matrix T:
Characteristic polynomial:
Exact first-order gauge H:
First-order gauge residual: zero in all 9 entries.
lambda = 3.259992875874317641659696207940516079827e-29 ; power of n = 0.556365409481672126
lambda = 0.2648235932673648722292552079638114764193 - 0.1922234742619173842910063619872265088373*I ; power of n = 0.556365409481672126
lambda = 0.2648235932673648722292552079638114764193 + 0.1922234742619173842910063619872265088373*I ; power of n = 0.556365409481672126
r = 3.259992875874317641659696207940516079827182785306938886e-29
R = 0.3272329439533261852269393658588798450146118026595228069
E = log(R/r) = 64.47615969189776924414743966571117329394928449421936402


Matrix([
[                         1, 67935889627/15298884797,   288899710760/15298884797],
[  -4289038063/611955391880,  532385237/611955391880,       419064958/2185554971],
[-67935889627/2447821567520, -7222492769/61195539188, -288367325523/611955391880]])

x**3 - 162060225797*x**2/305977695940 + 40100849267287098349209*x/374489401651004369934400 - 25/7161587766945221818712445059146

Matrix([
[107/660,    -13/396,  -59/2640],
[323/198, -2483/7920, -101/1760],
[      0,          0,         0]])

In [9]:
# Every right eigenvector of T has a nonzero first coordinate.
e=Matrix([[1,0,0]])
observation=Matrix.vstack(e,e*T,e*T*T)
obsdet=sp.factor(observation.det())
assert obsdet==sp.Rational(22488133988255638843697971020794,
                          3580793883472610909356222529573)
assert obsdet != 0
print('Terminal observability determinant =',obsdet)
print('Readout determinant det[D;X;Y] =', Matrix.vstack(D,X,Y).det())

# Contrast the full forward spectrum with its active rank-three image.
nf=Matrix(4,4,lambda i,j:sp.cancel(n**i*M[i,j]/M[0,0]/(n+1)**j))
Tf=SeriesMatrix.from_matrix(nf,var=n,p=1,precision=1).coeffs[0]
forward_poly=sp.Poly(sp.Matrix(Tf).charpoly(x).as_expr(),x)
forward_roots=sp.nroots(forward_poly,n=70,maxsteps=1000)
forward_moduli=sorted([abs(complex(z)) for z in forward_roots],reverse=True)
print('Full forward root moduli:',forward_moduli)
print('Full-forward top/second gap:',math.log(forward_moduli[0]/forward_moduli[1]))
print('Active pairing gap:',math.log(forward_moduli[0]/forward_moduli[2]))


Terminal observability determinant = 22488133988255638843697971020794/3580793883472610909356222529573
Readout determinant det[D;X;Y] = -117573120
Full forward root moduli: [4.819107166359313, 2.461177670058748e-14, 4.80093930660203e-28, 4.80093930660203e-28]
Full-forward top/second gap: 32.90814001456893
Active pairing gap: 64.47615969189776


### Why these modes give an error bound, not just a formal prediction

The exact first-order gauge reduces the transposed system to
$(1+b/n)T^T+O(n^{-2})$. Divide by the scalar product of $1+b/n$, whose
growth is $n^b$. The remainder is summable. Asymptotic integration of this
constant, diagonalizable system supplies a fundamental **row** matrix
\[
\Psi_N=\operatorname{diag}(\lambda_0^N N^b,
\lambda_1^N N^b,\lambda_2^N N^b)(V+o(1)),
\]
where the rows of $V$ are independent left eigenvectors of $T$,
$\lambda_0=r>0$, and $|\lambda_1|=|\lambda_2|=R>r$.
Equal moduli of the complex pair cause no difficulty: in that block the
diagonal constant dynamics and its inverse have bounded norm after removing
the common modulus, and the perturbation is summable. The separated real
mode is handled by the usual stable/unstable asymptotic integration.

Normalize the original right product starting at $N=1$, so the singular
gauge $D_g(0)$ is never used. Its fundamental matrix is $P\Psi_N$ for
an invertible **constant** matrix $P$. Consequently its inverse transpose
applied to $e_0$ is
\[
P^{-T}\operatorname{diag}(\lambda_i^{-N}N^{-b})
(V+o(1))^{-T}e_0.
\]
The observation determinant checked above says that the coefficient of the
$r^{-N}$ term tends to a nonzero value. Indeed its vanishing would give a
right eigenvector of $T$ whose first coordinate is zero, contradicting that
determinant. Restoring the scalar and polynomial gauges changes common
factors but not the ratio exponent: \(D_g(N)^T e_0=e_0\), so the terminal
gauge introduces no depth-dependent output direction. Thus
\[
v_N=a_N\bigl(u+O((r/R)^N e^{o(N)})\bigr),
\]
for a fixed nonzero vector $u$, with its entire dominant scalar correction
absorbed into $a_N$. This is why algebraic $1/N$ corrections to a single
mode do not create a $1/N$ error in these fixed readout ratios.

The vector $u$ is proportional to the initial recessive row, transposed:
the first row of $P^{-1}$ selects the recessive solution from the original
fundamental matrix. Section 6A constructs that solution explicitly and proves
$Du=3888$, $Xu=3888C$, $Yu=3888Z$ after choosing its normalization.
Consequently
\[
|r_C-C|+|r_Z-Z|\le \exp[-EN+o(N)],\qquad E=\log(R/r).
\]
Every **fixed** $(A,B)$ inherits this upper bound. Cancellations may improve an
individual error; equality of the error exponent is neither claimed nor needed.

The following integral argument supplies the initial-state identification needed
here, independently of the arithmetic conclusions.


## 6A. Analytic completion: the initial MG state and its recessiveness

This proves the two limit identifications used in the arithmetic argument.
The gamma/beta integral argument follows the construction in
Appendix B.1 of arXiv:2610.27799 (the zeta(3), zeta(4) recessiveness lemma).
No irrationality or linear-independence assumption is used.

### 1. The actual special-function state

Let theta = z d/dz and
\[
G_n(z)=G_{5,5}^{5,2}\!\left(
z\;\middle|\;
\begin{matrix}
-1-10n,-1-8n;\ 2+12n,7/3+6n,5/2+5n\\
0,n,2n,1/3,2n+1/2
\end{matrix}\right).
\]
Set $\mathcal G_n=(G_n(1),(\theta G_n)(1),(\theta^2G_n)(1))$.
The contiguous identities defining the Meijer-G CMF, and the zero lower-left
block of its four-dimensional trajectory matrix, give
\[
\mathcal G_{n+1}=\mathcal G_n Q(n).
\]
This is the special-function solution of the same restricted matrix used in
the arithmetic calculation.

On the upward line $\Re w=1/2$, the Mellin--Barnes formula is
\[
G_n(z)=\frac1{2\pi i}\int
\frac{\Gamma(w)\Gamma(w+n)\Gamma(w+2n)\Gamma(w+1/3)
\Gamma(w+2n+1/2)\Gamma(10n+2-w)\Gamma(8n+2-w)}
{\Gamma(w+12n+2)\Gamma(w+6n+7/3)\Gamma(w+5n+5/2)}
z^{-w}\,dw .
\]
The line separates all left and right poles for every integer $n\ge0$.
For $z>0$, this integral and its first two theta derivatives converge
absolutely. Along the vertical line the gamma factors give exponential
decay in the imaginary direction.

### 2. A positive integral in Laplace form

Apply Euler's beta integral to the three ratios
\[
\frac{\Gamma(w+2n)}{\Gamma(w+12n+2)},\quad
\frac{\Gamma(w+1/3)}{\Gamma(w+6n+7/3)},\quad
\frac{\Gamma(w+2n+1/2)}{\Gamma(w+5n+5/2)}.
\]
Their beta differences are $10n+2,6n+2,3n+2$.
For the fourth integration use
\[
\Gamma(w+n)\Gamma(8n+2-w)
=\Gamma(9n+2)\int_0^1 r^{w+n-1}(1-r)^{8n+1-w}\,dr.
\]
Every beta parameter has positive real part on the chosen contour.
Absolute convergence permits interchanging these integrations.
Writing $u=xyt$ and $H_z=z(1-r)+ur$, Mellin inversion gives
\[
\frac1{2\pi i}\int
\Gamma(w)\Gamma(10n+2-w)
\left(\frac{z(1-r)}{ur}\right)^{-w}dw
=\Gamma(10n+2)\left(\frac{ur}{H_z}\right)^{10n+2}.
\]
Consequently
\[
\boxed{
G_n(z)=\frac{\Gamma(9n+2)}{\Gamma(6n+2)\Gamma(3n+2)}
\int_{[0,1]^4}
\frac{
x^{12n+1}y^{10n+4/3}t^{12n+3/2}r^{11n+1}
(1-x)^{10n+1}(1-y)^{6n+1}(1-t)^{3n+1}(1-r)^{8n+1}
}{[z(1-r)+xytr]^{10n+2}}\,dx\,dy\,dt\,dr .
}
\]
In particular, $G_n(1)>0$.

This is a Laplace integral. At $z=1$, let $H=1-r+xytr$ and
\[
\Phi=
\frac{x^{12}y^{10}t^{12}r^{11}
(1-x)^{10}(1-y)^6(1-t)^3(1-r)^8}{H^{10}},
\quad
\omega=\frac{x\,y^{4/3}t^{3/2}r(1-x)(1-y)(1-t)(1-r)}{H^2}.
\]
Then $G_n(1)=\Gamma(9n+2)/(\Gamma(6n+2)\Gamma(3n+2))
\int\omega\,\Phi^n$.
An exact saddle evaluation is unnecessary to identify the recessive mode.
The following elementary beta bound already separates it from both other modes.

### 3. An exponential upper bound for the whole state

Since $H- xyt=(1-r)(1-xyt)\ge0$, replacing $H$ by $xyt$ in the
denominator gives, for $n\ge1$,
\[
\begin{aligned}
0<G_n(1)\le\beta_n
&=\frac{\Gamma(9n+2)}{\Gamma(6n+2)\Gamma(3n+2)}
B(2n,10n+2)B(1/3,6n+2)\\
&\qquad {}\cdot B(2n+1/2,3n+2)B(11n+2,8n+2)\\
&=\frac{\Gamma(1/3)\Gamma(9n+2)\Gamma(2n)\Gamma(10n+2)
\Gamma(2n+1/2)\Gamma(11n+2)\Gamma(8n+2)}
{\Gamma(12n+2)\Gamma(6n+7/3)\Gamma(5n+5/2)\Gamma(19n+4)}.
\end{aligned}
\]
Gamma recurrence, or Stirling's formula, gives the exact ratio limit
\[
\eta=\lim_{n\to\infty}\frac{\beta_{n+1}}{\beta_n}
=\frac{9^9\,2^4\,10^{10}\,11^{11}\,8^8}
{12^{12}\,6^6\,5^5\,19^{19}}
=\frac{228249336488800000}{1978419655660313589123979}
<10^{-6}.
\]
Numerically, $\eta=1.15369525285382354677\ldots\,10^{-7}$.
Thus $\beta_n=\exp(n\log\eta+o(n))$.

To bound derivatives, put $a=10n+2$ and $h=z(1-r)/H_z$, so $0\le h\le1$.
On the positive integrand, theta differentiation multiplies by
\[
-ah,\qquad a(a+1)h^2-ah
\]
for the first and second derivatives respectively. Differentiation under the
integral is justified on compact positive $z$ intervals by domination.
Therefore, at $z=1$,
\[
|\theta G_n(1)|\le a\beta_n,\qquad
|\theta^2G_n(1)|\le a(a+2)\beta_n.
\]
All three components are exponentially small, and the state is nonzero because
its first component is positive. This is a bound on the entire state, not just
one readout.

### 4. Identification of the recessive spectral mode

In the notebook's normalization,
\[
T=\lim_{n\to\infty}
\frac{D_g(n)^{-1}Q(n)D_g(n+1)}{Q_{00}(n)},
\quad D_g(n)=\operatorname{diag}(1,n^{-1},n^{-2}),
\quad \lim Q_{00}(n)=s_\infty=156660580321280.
\]
The three simple roots of $T$ are $r,\lambda,\bar\lambda$, with
\[
r=3.25999287587431764\ldots\,10^{-29},\qquad
|\lambda|=R=0.32723294395332618\ldots.
\]
The actual, unscaled exponential root moduli of the state recurrence are
\[
s_\infty r=5.1071237577770912\ldots\,10^{-15},
\qquad
s_\infty R=5.1264502899968973\ldots\,10^{13}
\quad\text{(twice)}.
\]
Rational root isolation verifies
$0<s_\infty r<\eta<1<s_\infty R$.
The exact first-order gauge and the summable-perturbation argument in the
notebook supply a fundamental solution basis with these exponential moduli.
Polynomial gauges do not change exponential growth. A nonzero component in
the two-dimensional growing sector would have norm
$(s_\infty R)^{n+o(n)}$; oscillations of the conjugate roots cannot cancel
the whole state vector because their asymptotic eigenvectors are independent.
The bound in Section 3 rules out that sector. Hence $\mathcal G_n$ is the
nonzero solution in the one-dimensional recessive sector.

### 5. Exact evaluation at the initial point

With $s=-w$, the Barnes kernel at $n=0$ is $\pi^2\csc^2(\pi s)F_0(s)$, where
\[
F_0(s)=\frac{(s+1)^2}
{(-s)(1-s)(1/3-s)(4/3-s)(1/2-s)(3/2-s)}.
\]
For any $\rho\ge0$, closing the contour $\Re s=-1/2$ to the left gives
\[
\frac1{2\pi i}\int
\frac{\pi^2\csc^2(\pi s)}{\rho-s}\,ds
=\sum_{m\ge1}\frac1{(\rho+m)^2}
=\psi_1(\rho+1).
\]
The left vertical edge tends to zero and the horizontal edges vanish
exponentially. The orientation here is fixed, giving the positive sign.
The exact partial fraction expansions of $s^kF_0(s)$, $k=0,1,2$, therefore
evaluate the three initial moments.

Using
$\psi_1(1)=Z$, $\psi_1(1/3)=4Z+9C/2$, and $\psi_1(1/2)=3Z$,
which follow directly by splitting their defining reciprocal-square series,
the result is
\[
\boxed{
\mathcal G_0=
\left(
\frac{14823}{70}C+\frac{6477}{35}Z-\frac{789349}{1680},\
\frac{16362}{35}C+\frac{8676}{35}Z-\frac{108221}{140},\
\frac{23976}{35}C+\frac{10368}{35}Z-\frac{35787}{35}
\right).
}
\]
Equivalently,
$\mathcal G_0=(1,9C/2+3Z,2Z)\widehat B_0$ with the endpoint matrix already
derived in the notebook.
The exact readout identities are
\[
D\mathcal G_0^T=3888,\qquad
X\mathcal G_0^T=3888C,\qquad
Y\mathcal G_0^T=3888Z.
\]
In particular the limiting denominator is nonzero, without assuming any
irrationality or independence.

### 6. Dual convergence and the identified limits

The checked observation determinant
\[
\det\begin{pmatrix}e_0^T\\e_0^TT\\e_0^TT^2\end{pmatrix}
=\frac{22488133988255638843697971020794}
{3580793883472610909356222529573}\ne0
\]
shows that the terminal $e_0$ sees the minimal mode in the inverse-transpose
product. With the right-product convention, dual convergence recovers the
transpose of the initial recessive row. Thus, for suitable nonzero scalars $a_N$,
\[
W_N^{-T}e_0
=a_N\left(\mathcal G_0^T+
O\left(e^{-EN+o(N)}\right)\right),
\qquad E=\log(R/r)=64.47615969189776924414\ldots.
\]
Here $D_g(N)^Te_0=e_0$, so the terminal gauge causes no moving readout.
Taking the three exact pairings above proves, without a limit assumption,
\[
\frac{XW_N^{-T}e_0}{DW_N^{-T}e_0}\longrightarrow C,\qquad
\frac{YW_N^{-T}e_0}{DW_N^{-T}e_0}\longrightarrow Z,
\]
and both errors are at most $\exp(-EN+o(N))$.
Every fixed rational linear combination inherits this upper error rate.

This proves both analytic inputs of the arithmetic
argument. The arithmetic calculation, its written prime-valuation lemmas,
and its numerical lower comparison are unchanged.

Sources for the standard integral identities:
https://dlmf.nist.gov/16.17.E1
https://dlmf.nist.gov/5.12.E1
https://dlmf.nist.gov/5.13.E1

The accompanying analytic_limit_certificate.py checks the finite rational
algebra, beta exponents and cancellations, root separation, and initial
readout identities. Barnes quadrature at 50 and 80 digits is included only
as an independent regression; no numerical quadrature is used in the proof.


In [10]:
analytic_source = r'''#!/usr/bin/env python3
"""Exact algebraic checks for the initial state and recessive MG integral.

The analytic argument (beta integrals, absolute convergence and asymptotic
integration) is written in ANALYTIC_PROOF.txt. Optional Barnes quadrature is
an independent numerical regression, not a premise of that argument.
"""
import argparse, json, time
from pathlib import Path
import sympy as s
from flint import arb, ctx

def verify(matrix_file, quadrature=False):
    begun=time.monotonic()
    n,w,t=s.symbols('n w t')
    C,Z=s.symbols('C Z')
    data=json.loads(Path(matrix_file).read_text())
    Q=s.Matrix([[s.sympify(z,locals={'n':n}) for z in row]
                 for row in data['MG55']])[:3,:3]
    D=s.Matrix([[243648,-498096,264600]])
    X=s.Matrix([[190344,-389162,206743]])
    Y=s.Matrix([[400824,-819330,435234]])

    # Derive Bhat_0 directly from the six simple poles; no fitted constants.
    F0=(t+1)**2/((-t)*(1-t)*(s.Rational(1,3)-t)*(s.Rational(4,3)-t)
                 *(s.Rational(1,2)-t)*(s.Rational(3,2)-t))
    groups=[(s.S.Zero,[0,1]),(s.Rational(1,3),[0,1]),(s.Rational(1,2),[0,1])]
    rows=s.zeros(4,3); decomp=[0,0,0]
    for cl,(offset,indices) in enumerate(groups):
        for j in indices:
            rho=offset+j
            v=-s.cancel((t-rho)*F0).subs(t,rho)
            h=sum((1/(offset+l)**2 for l in range(1 if cl==0 else 0,j+1)),s.S.Zero)
            for k in range(3):
                rows[0,k]-=v*h*rho**k
                rows[cl+1,k]+=v*rho**k
                decomp[k]+=v*rho**k/(rho-t)
    assert all(s.cancel(t**k*F0-decomp[k])==0 for k in range(3))
    assert all(sum(rows[j,k] for j in (1,2,3))==0 for k in range(3))
    Bhat=s.Matrix([list(rows.row(j)) for j in (0,2,3)])
    state=s.Matrix([[1,s.Rational(9,2)*C+3*Z,2*Z]])*Bhat
    assert (D*state.T)[0]==3888
    assert s.expand((X*state.T)[0])==3888*C
    assert s.expand((Y*state.T)[0])==3888*Z

    # Three beta ratios and the fourth beta integral recover the Barnes kernel.
    pairs=[(2*n,10*n+2),(s.Rational(1,3),6*n+2),
           (2*n+s.Rational(1,2),3*n+2)]
    kernel=(s.gamma(w)*s.gamma(w+n)*s.gamma(w+2*n)*s.gamma(w+s.Rational(1,3))
            *s.gamma(w+2*n+s.Rational(1,2))*s.gamma(10*n+2-w)*s.gamma(8*n+2-w)
            /(s.gamma(w+12*n+2)*s.gamma(w+6*n+s.Rational(7,3))
              *s.gamma(w+5*n+s.Rational(5,2))))
    beta_product=s.prod(s.gamma(w+a)/s.gamma(w+a+b) for a,b in pairs)
    assert s.cancel(kernel-beta_product*s.gamma(w)*s.gamma(w+n)
                    *s.gamma(10*n+2-w)*s.gamma(8*n+2-w))==0
    assert s.expand((w+n)+(8*n+2-w))==9*n+2
    mellin_a=10*n+2
    numerator_powers=[s.expand(a-1+mellin_a) for a,b in pairs]+[11*n+1]
    assert numerator_powers==[12*n+1,10*n+s.Rational(4,3),12*n+s.Rational(3,2),11*n+1]
    prefactor=s.gamma(9*n+2)/(s.gamma(6*n+2)*s.gamma(3*n+2))
    bound_pairs=[(2*n,10*n+2),(s.Rational(1,3),6*n+2),
                 (2*n+s.Rational(1,2),3*n+2),(11*n+2,8*n+2)]
    beta_bound=prefactor*s.prod(s.gamma(a)*s.gamma(b)/s.gamma(a+b) for a,b in bound_pairs)
    gamma_bound=(s.gamma(s.Rational(1,3))*s.gamma(9*n+2)*s.gamma(2*n)
                 *s.gamma(10*n+2)*s.gamma(2*n+s.Rational(1,2))*s.gamma(11*n+2)*s.gamma(8*n+2)
                 /(s.gamma(12*n+2)*s.gamma(6*n+s.Rational(7,3))
                   *s.gamma(5*n+s.Rational(5,2))*s.gamma(19*n+4)))
    assert s.cancel(beta_bound-gamma_bound)==0
    eta=s.Rational(9**9*2**4*10**10*11**11*8**8,12**12*6**6*5**5*19**19)
    assert 9+2+10+2+11+8==12+6+5+19
    assert 0<eta<s.Rational(1,1000000)

    # Bounds on theta derivatives of H_z^(-a), using 0<=h<=1.
    a,h=s.symbols('a h')
    first=-a*h; second=s.expand(first**2-a*h*(1-h))
    assert s.expand(second-(a*(a+1)*h**2-a*h))==0
    derivative_bounds=[s.S.One,mellin_a,s.expand(mellin_a*(mellin_a+2))]

    # Actual unscaled roots: the entire state decays, the nonminimal modes grow.
    scale=s.limit(Q[0,0],n,s.oo)
    assert scale==156660580321280
    T=s.Matrix(3,3,lambda i,j:s.limit(n**i*Q[i,j]/Q[0,0]/(n+1)**j,n,s.oo))
    x=s.Symbol('x');p=s.Poly(T.charpoly(x).as_expr(),x)
    assert p.all_coeffs()==[1,-s.Rational(162060225797,305977695940),
        s.Rational(40100849267287098349209,374489401651004369934400),
        -s.Rational(25,7161587766945221818712445059146)]
    lo=s.Rational('3.25999287587431764165969620794e-29')-s.Rational('1e-58')
    hi=lo+2*s.Rational('1e-58')
    assert p.eval(lo)<0<p.eval(hi) and s.discriminant(p.as_expr(),x)<0
    assert scale*hi<eta<1
    assert scale**2*(-p.all_coeffs()[-1])/hi>1
    e=s.Matrix([[1,0,0]])
    observation=s.Matrix.vstack(e,e*T,e*T*T).det()
    assert observation!=0

    ctx.prec=256
    ball=lambda z:arb(int(s.numer(z)))/int(s.denom(z))
    real_interval=ball(lo).union(ball(hi))
    large_mod=ball(scale)*(-ball(p.all_coeffs()[-1])/real_interval).sqrt()
    initial_values=[]
    cv=(arb(2).zeta(arb(1)/3)-arb(2).zeta(arb(2)/3))/9;zv=arb(2).zeta()
    for entry in state:
        initial_values.append(ball(entry.coeff(C))*cv+ball(entry.coeff(Z))*zv+ball(entry.subs({C:0,Z:0})))
    assert initial_values[0]>0
    regressions=[]
    if quadrature:
        import mpmath as mp
        for dps in (50,80):
            mp.mp.dps=dps
            cc=(mp.zeta(2,mp.mpf(1)/3)-mp.zeta(2,mp.mpf(2)/3))/9;zz=mp.zeta(2)
            for k in range(3):
                def integrand(y):
                    ss=mp.mpc(-.5,y)
                    f=(ss+1)**2/((-ss)*(1-ss)*(mp.mpf(1)/3-ss)*(mp.mpf(4)/3-ss)
                                   *(mp.mpf(1)/2-ss)*(mp.mpf(3)/2-ss))
                    return mp.pi*mp.re(ss**k*f)/mp.cosh(mp.pi*y)**2
                integral=mp.quad(integrand,[0,1,3,8,mp.inf])
                exact=mp.mpf(str(s.N(state[k].subs({C:str(cc),Z:str(zz)}),dps)))
                difference=abs(integral-exact)
                assert difference<mp.mpf(10)**(-dps+6)
                regressions.append(dict(dps=dps,k=k,barnes=mp.nstr(integral,35),
                                        absolute_difference=mp.nstr(difference,8)))
    return dict(algebraic_checks_passed=True,initial_state=[str(z) for z in state],
        initial_state_values=[z.str(30) for z in initial_values],
        denominator_initial_pairing='3888',C_initial_pairing='3888*C',Z_initial_pairing='3888*Z',
        positive_integral_prefactor=str(prefactor),integral_numerator_exponents=list(map(str,numerator_powers)),
        beta_bound=str(gamma_bound),beta_exponential_base=str(eta),
        beta_exponential_base_decimal=ball(eta).str(30),log_beta_base=ball(eta).log().str(30),
        derivative_bounds=list(map(str,derivative_bounds)),unscaled_small_root=(ball(scale)*real_interval).str(30),
        unscaled_other_root_modulus=large_mod.str(30),observation_determinant=str(observation),
        analytic_assumptions_remaining=[],
        interpretation='The written positive-integral argument proves recessiveness; exact initial values identify both dual limits.',
        quadrature_role='Independent numerical regression only; no quadrature is used as a proof premise.',
        quadrature_regressions=regressions,seconds=time.monotonic()-begun)

if __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--matrices',type=Path)
    parser.add_argument('--output',type=Path)
    parser.add_argument('--quadrature',action='store_true')
    args=parser.parse_args()
    folder=Path(__file__).resolve().parent
    matrices=args.matrices or (folder/'notebook_outputs/exact_matrices.json')
    result=verify(matrices,args.quadrature)
    target=args.output or folder/'analytic_limit_result.json'
    target.write_text(json.dumps(result,indent=2)+'\n')
    print(json.dumps(result,indent=2))
'''
analytic_path = OUT/'analytic_limit_certificate.py'
analytic_path.write_text(analytic_source)
print('Analytic source SHA256:', hashlib.sha256(analytic_path.read_bytes()).hexdigest())


Analytic source SHA256: ef760ef7efe2e1a5004a6bd2fad5d35b765f792d1774fb66e29ce74daaf32791


In [11]:
import importlib.util
analytic_spec=importlib.util.spec_from_file_location('analytic_cert',analytic_path)
analytic_module=importlib.util.module_from_spec(analytic_spec)
analytic_spec.loader.exec_module(analytic_module)
analytic_result=analytic_module.verify(OUT/'exact_matrices.json',quadrature=True)
assert analytic_result['algebraic_checks_passed']
assert analytic_result['analytic_assumptions_remaining']==[]
assert Matrix([analytic_result['initial_state']]).applyfunc(sp.sympify) == (
    Matrix([[1,sp.Rational(9,2)*sp.Symbol('C')+3*sp.Symbol('Z'),2*sp.Symbol('Z')]])*B0)
(OUT/'analytic_limit_result.json').write_text(json.dumps(analytic_result,indent=2)+'\n')
print('Initial state:',analytic_result['initial_state'])
for key in ['denominator_initial_pairing','C_initial_pairing','Z_initial_pairing',
            'beta_exponential_base','beta_exponential_base_decimal',
            'unscaled_small_root','unscaled_other_root_modulus']:
    print(key,':',analytic_result[key])
print('Barnes regression precisions: 50 and 80 decimal digits, all three moments.')
print('Both limit identifications now follow from the written integral argument.')


Initial state: ['14823*C/70 + 6477*Z/35 - 789349/1680', '16362*C/35 + 8676*Z/35 - 108221/140', '23976*C/35 + 10368*Z/35 - 35787/35']
denominator_initial_pairing : 3888
C_initial_pairing : 3888*C
Z_initial_pairing : 3888*Z
beta_exponential_base : 228249336488800000/1978419655660313589123979
beta_exponential_base_decimal : [1.15369525285382354677061439612e-7 +/- 2.10e-37]
unscaled_small_root : [5.1071237577770912000754094247e-15 +/- 3.68e-44]
unscaled_other_root_modulus : [51264502899968.9733401643894700 +/- 7.92e-17]
Barnes regression precisions: 50 and 80 decimal digits, all three moments.
Both limit identifications now follow from the written integral argument.


## 7. Arithmetic: raw height, common content, and actual reduced denominator

For the rational cofactor vector let $g_N=\operatorname{cont}(w_N)$ and
$u_N=w_N/g_N\in\mathbb Z^3$ be primitive. The actual denominator is
\[
q_N(A,B)=\frac{9|u_0|}{\gcd(9u_0,\,2((A+B)u_1-3Bu_2))}\le9|u_0|.
\]
The triple content is only a sufficient bound shared by all mixtures.
The two-integer gcd determines each actual reduced denominator.

The positive forward residue sum gives raw cofactor height
$\alpha_{\rm raw}=63.136039354657680679\ldots$; its unique saddle is the
root in $(6,7)$ of $-180y^3-720y^2+6480y+28800=0$.
The forward/dual normalization is the exact determinant gamma product checked next.
The arithmetic lemmas and all prime ranges are reproduced in the appendix.


In [12]:
gamma_terms=[(12,sp.Rational(2,3),-1),(8,1,1),(5,sp.Rational(1,3),-1),
 (8,sp.Rational(1,3),1),(6,1,-1),(5,1,1),(4,sp.Rational(1,6),1),
 (9,sp.Rational(1,2),-1),(6,sp.Rational(2,3),1),(5,sp.Rational(2,3),1),
 (4,sp.Rational(1,3),-1)]
g=(3936600*(8*n+1)*(9*n+1)*(15*n+2)*(20*n+3)*(24*n+1)**2
 /((2*n+1)*(3*n+1)**2*(6*n+1)*(8*n+3)*(9*n+2)*(10*n+3)
   *(12*n+1)*(15*n+1)*(15*n+4)*(18*n+1)**2*(24*n+5)*(30*n+7)))
detstep=2**8*3**30*g.subs(n,n+1)/g*sp.prod(sp.rf(a*n+b,a)**m for a,b,m in gamma_terms)
assert sp.cancel(Q.det()-detstep)==0
assert sum(a*m for a,b,m in gamma_terms)==0
print('Exact determinant/gamma normalization identity verified.')

# Pairwise reduced denominators are checked independently of the displayed formulas.
for N,w in moments.items():
    content=qcontent(w); primitive=[int(z/content) for z in w]
    assert all(z/content==int(z/content) for z in w)
    assert math.gcd(*primitive)==1
    for _,ab in cases:
        if ab is None:continue
        a,b=ab; d=9*abs(primitive[0]); h=2*((a+b)*primitive[1]-3*b*primitive[2])
        assert int(mix(w,a,b).denominator)==d//math.gcd(d,h)
print('Actual scalar denominator formula checked for every recorded mixture and depth.')


Exact determinant/gamma normalization identity verified.
Actual scalar denominator formula checked for every recorded mixture and depth.


### Finite chamber certificate and the remaining prime ranges

The verifier uses **1,158 exact dual chambers**: 264 at lift 0, 264 at lift 1,
and 35 improving chambers at lifts 2 through 19. It takes the maximum of
the forward and dual lower content bounds at each prime. It does not add them.

- For large primes with $x=N/p<20$, it enumerates the exact rational chamber
  boundaries, unit Taylor generators, and the intersection with the infinity-residue constraint.
- For $x\ge20$, it uses the elementary forward bound and its proved count periodicity.
  It does **not** extrapolate the dual profile periodically.
- Small primes $5\le p\le\sqrt{132N+30}$, affine boundary exceptions, and rational
  coboundaries cost $o(N)$.
- The primes 2 and 3 contribute content slopes at least 12 and 18 respectively.

The prime density is summed exactly on the finite range and with an Arb digamma
interval for the infinite tail. The complete source is embedded below so the
notebook is runnable without the original workspace. The source cell is collapsed
for readability; it can be expanded and audited.


In [13]:
arithmetic_source = r'''#!/usr/bin/env python3
"""Conditional positive-delta arithmetic certificate for the specified MG55 ray.

Requires Python 3.10+ and python-flint. No numerical target values or observed
prime valuations are inputs. Default run recomputes 1,158 exact dual chambers
and the forward bounds; about six minutes on the development machine.

The mathematical lemmas justifying this finite calculation are in PROOF.txt.
Only the limit and error rate are analytic assumptions. Interrupted runs resume
from a checkpoint bound to the SHA256 of this file; --fresh discards that cache.
"""
import argparse,hashlib,json,math,time
from pathlib import Path
from fractions import Fraction as F
from functools import lru_cache
from flint import fmpq as Q, fmpq_mat as Mat, arb, ctx
J=5


GAMMA=[[12, '2/3', -1], [8, '1', 1], [5, '1/3', -1], [8, '1/3', 1], [6, '1', -1], [5, '1', 1], [4, '1/6', 1], [9, '1/2', -1], [6, '2/3', 1], [5, '2/3', 1], [4, '1/3', -1]]

GAMMA_CONSTANT=52708129816230144

NONVANISHING={'scope': 'Exact nonzero evaluation of the rational three-consecutive-column determinant. Therefore it vanishes at only finitely many n; together with the fixed readout determinant this supplies infinitely many nonzero errors for every nonzero fixed mixture.', 'Q0': [['33134470494113265094822158549/101544012423478625', '1149023549983998483051406866/1562215575745825', '-2260235960860916506330466676/600852144517625'], ['3381503995827857395882195659/10977731072808500', '190120812262917053411793303/84444085175450', '255280621821754797650639421/16239247149125'], ['40918696764765654021173289/663144570928840', '6435216689897315510132013/5101112084068', '12929524885498886211929391/980983093090']], 'Q1': [['2333587742808374761877390652393461240832/142477313395059410047039375', '138193876084407075096438790730174379264/884952257112170248739375', '8607877940978977256442996241789944101376/6194665799785191741175625'], ['14177626252700960086117268790801413284128/101871279077467478183633153125', '2006365731574024205959048565259297358656/632740863835201727848653125', '236319967208750935311864334743377942680704/4429186046846412094940571875'], ['-5425557236209125232650198139313520476208/20374255815493495636726630625', '-290290269235070602777106726268103686816/126548172767040345569730625', '-15122547257964799477651558650507637767744/885837209369282418988114375']], 'determinant_at_n0': '-2091300253601094307522404926786910050752593888802048297/255217484616350899365532475817984000', 'native_matrix_sha256': 'a9b1b0fa4630fd74092c1d42f6c8da70ef9272a11b4b116021a2465e814026c9', 'readout_determinant': '-117573120'}

FORWARD_INITIAL={'F0': [['41/3', '35/3', '35/3', '35/3'], ['1807/16', '1339/16', '8495/144', '37819/1296'], ['2353/54', '818/27', '3649/216', '311/432']], 'F0K0': [['-188/3', '1153/9', '-1225/18'], ['-6355/12', '155897/144', '-55211/96'], ['-5567/27', '136555/324', '-72539/324']], 'nine_residuals_zero': True}

def walls(e):
    t=F(2,3) if e==1 else F(1,3)
    lines=[(-10,F(0)),(0,F(0)),(-8,F(0)),(1,F(0)),
           (2,F(0)),(12,F(0)),(0,t),(6,t),(2,F(1,2)),(5,F(1,2))]
    cuts={F(0),F(1)}
    for u,a in lines:
        for v,b in lines:
            if u==v:continue
            for k in range(-25,26):
                x=(b-a+k)/(u-v)
                if 0<x<1:cuts.add(x)
    return sorted(cuts)

def projected_bound(gens,which,A=None,B=None):
    first=None
    for weight,v in sorted(gens,key=lambda z:z[0]):
        if which=='TH':u=(v[1],v[2])
        elif which=='CT':u=(v[0],v[1])
        elif which=='CH':u=(v[0],v[2])
        else:u=(v[0],(A+B)*v[2]+3*B*v[1])
        if not any(u):continue
        if first is None:first=(weight,u);continue
        w,a=first
        if a[0]*u[1]-a[1]*u[0]:return w+weight
    return 10**6

def intervals(e):
    theta=F(1,3) if e==1 else F(2,3)
    return [(-10,0,F(-1),F(-1),F(0)),(-8,1,F(-1),F(-1),F(0)),
            (2,12,F(0),F(1),F(0)),(0,6,F(1,3),F(4,3),theta),
            (2,5,F(1,2),F(3,2),F(1,2))]

def signatures(a,b,e,L=0):
    x=a+(b-a)*F(503,1009)+L;data=intervals(e)
    cuts={F(0),F(1)};boundaries=[]
    for u,v,off0,off1,theta in data:
        for slope in (u,v):
            for h in range(math.floor(slope*x-theta)-1,math.ceil(slope*x-theta)+1):
                y=slope*x-h-theta
                if 0<=y<=1:
                    cuts.add(y);boundaries.append((y,off0%1))
    cuts=sorted(cuts);packets=set()
    def add(y,z=None):
        lists=[]
        for u,v,a0,b0,theta in data:
            lo=u*x-y;hi=v*x-y
            q0=math.ceil(lo-theta);q1=math.floor(hi-theta)
            if q0+theta==lo and not(z is not None and z>=a0):q0+=1
            if q1+theta==hi and not(z is not None and z<=b0):q1-=1
            lists.append(tuple(F(q)+theta for q in range(q0,q1+1)))
        ns=tuple(sorted(lists[0]+lists[1]));ds=tuple((t,c) for roots,c in zip(lists[2:],(0,1,2)) for t in roots)
        packets.add((ns,ds,int(y==0 and z==0)))
    for u,v in zip(cuts,cuts[1:]):add((u+v)/2)
    # r=0,1,p-1 additionally matter for the terminal polynomial s.
    boundaries.extend([(F(0),F(0)),(F(1),F(0))])
    for y,off in boundaries:
        for z in [F(i)+off for i in range(-2,3)]:
            if(y==0 and z<0)or(y==1 and z>=0):continue
            add(y,z)
    return packets

@lru_cache(None)
def harmonic(t):
    r=t%1;j=math.floor(t)
    if r==0:return sum((Q(1,i*i) for i in range(1,j+1)),Q(0))
    return sum((Q(r.denominator**2,(r.denominator*i+r.numerator)**2) for i in range(j+1)),Q(0))

@lru_cache(None)
def vectors(ns,ds,jmin):
    if not ds:return ()
    h=len(ns)-len(ds);rs=[]
    for t,c in ds:
        r=Q(1)
        for u in ns:r*=Q((t-u).numerator,(t-u).denominator)
        for u,_ in ds:
            if u!=t:r/=Q((t-u).numerator,(t-u).denominator)
        rs.append((Q(t.numerator,t.denominator),c,r,harmonic(t)))
    out=[]
    for j in range(jmin,J+1):
        v=[Q(0)]*4
        for t,c,r,H in rs:
            z=r*t**j;v[0]-=z*H;v[3]+=z
            if c:v[c]+=z
        if any(v):out.append((h+1+j,tuple(v)))
    regular=min(h+1+max(jmin,-h-1),h+2+jmin)
    out.append((regular+2,(Q(1),Q(0),Q(0),Q(0))))
    for k in range(4):out.append((h+J+2,tuple(Q(int(i==k)) for i in range(4))))
    return tuple(out)

def cell(a,b,e,L=0):
    packets=signatures(a,b,e,L);gens=set()
    for ns,ds,jmin in packets:gens.update(vectors(ns,ds,jmin))
    tw,pivot=min(((w,v) for w,v in gens if v[3]),key=lambda z:(z[0],abs(z[1][3])))
    assert tw==0 and pivot[3] in (1,-1),(a,b,e,L,tw,pivot)
    gens={(w,tuple(v[k]-v[3]*pivot[k]/pivot[3] for k in range(3))) for w,v in gens}
    bounds=[projected_bound(gens,s)-(0 if s=='TH' else 2) for s in ('TH','CT','CH')]
    return dict(mod6=e,lift=L,lo=str(a),hi=str(b),bounds=bounds,common_bound=min(bounds),packet_count=len(packets))

def determinant_profile(x,e):
    val=0
    for A,b,m in GAMMA:
        b=F(b);d=b.denominator
        residue=(b.numerator*pow(e,-1,d))%d if d>1 else 0
        theta=F(residue,d) if residue else F(1)
        val+=m*math.floor(A*x+1-theta)
    return val

def highcounts(x,y,e,z=None):
    tt=F(2,3) if e==1 else F(1,3)
    data=[(-12,-2,F(0),F(-1),F(-1)),(-6,0,tt,F(-4,3),F(-4,3)),(-5,-2,F(1,2),F(-3,2),F(-3,2)),(0,10,F(0),F(0),F(1)),(-1,8,F(0),F(0),F(1))]
    nums=[];dens=[]
    for i,(u,v,theta,a0,b0) in enumerate(data):
        lo=math.ceil(u*x-y-theta);hi=math.floor(v*x-y-theta)
        if lo==u*x-y-theta and not(z is not None and z>=a0):lo+=1
        if hi==v*x-y-theta and not(z is not None and z<=b0):hi-=1
        roots=[F(j)+theta for j in range(lo,hi+1)]
        (nums if i<3 else dens).extend(roots)
    return nums,dens

def baseline(a,b,e,L=0):
    x=a+(b-a)*F(503,1009)+L;cuts={F(0),F(1)};boundaries=[]
    tt=F(2,3) if e==1 else F(1,3)
    for u,theta in [(-12,0),(-2,0),(-6,tt),(0,tt),(-5,F(1,2)),(-2,F(1,2)),(0,0),(10,0),(-1,0),(8,0)]:
        for h in range(math.floor(u*x-theta)-1,math.ceil(u*x-theta)+1):
            y=u*x-h-theta
            if 0<=y<=1:
                cuts.add(y)
                off=(F(theta)*e)%1
                boundaries.append((y,off))
    cuts=sorted(cuts);hmin=0
    for u,v in zip(cuts,cuts[1:]):
        ns,ds=highcounts(x,(u+v)/2,e)
        if ds:hmin=min(hmin,len(ns)-len(ds))
    bulk=hmin
    for y,off in boundaries:
        for z in [F(i)+off for i in range(-3,3)]:
            if(y==0 and z<0)or(y==1 and z>=0):continue
            ns,ds=highcounts(x,y,e,z)
            if ds:hmin=min(hmin,len(ns)-len(ds))
    return dict(forward_bulk_bound=bulk,forward_bound=hmin,determinant_profile=determinant_profile(x,e),combined_bulk_bound=bulk+determinant_profile(x,e),combined_bound=hmin+determinant_profile(x,e))


def ball(q):
    q=F(q);return arb(q.numerator)/q.denominator

def auxiliary_checks():
    assert sum(A*m for A,b,m in GAMMA)==0
    assert 10+6+3==10+9
    for e in (1,5):
        W=set(walls(e))
        for A,b,m in GAMMA:
            b=F(b);d=b.denominator
            r=(b.numerator*pow(e,-1,d))%d if d>1 else 0
            theta=F(r,d) if r else F(1)
            for j in range(A+2):
                x=(j-1+theta)/A
                assert not(0<x<1) or x in W
    # Exceptional primes. Legendre errors and rational coboundaries are O(log n).
    slopes=[]
    for p,forward,expected in [(2,-6,12),(3,-9,18)]:
        c=GAMMA_CONSTANT;v=0
        while c%p==0:c//=p;v+=1
        for A,b,m in GAMMA:
            den=F(b).denominator
            if den%p:
                vp=F(1,p-1)
            else:
                vp=0
                while den%p==0:den//=p;vp-=1
            v+=m*A*vp
        assert v+forward==expected
        slopes.append([p,forward,str(v),str(expected)])
    Q0=Mat([[Q(z) for z in r] for r in NONVANISHING['Q0']])
    Q1=Mat([[Q(z) for z in r] for r in NONVANISHING['Q1']])
    e0=Mat([[1],[0],[0]]);T0=Q0.inv().transpose();T1=Q1.inv().transpose()
    cols=[e0,T0*e0,T0*T1*e0]
    det=Mat([[c[j,0] for c in cols] for j in range(3)]).det()
    assert det==Q(NONVANISHING['determinant_at_n0']) and det
    D=[243648,-498096,264600];C=[190344,-389162,206743];Z=[400824,-819330,435234]
    assert Mat([D,C,Z]).det()==-117573120
    K0=Mat([[Q(11,3),Q(23,3),Q(-35,6)],
            [Q(-23,2),Q(4),Q(1)], [Q(-7,6),Q(-2),Q(0)], [Q(3),Q(0),Q(0)]])
    F0=Mat([[Q(z) for z in r] for r in FORWARD_INITIAL['F0']])
    want=Mat([[Q(x,-3888) for x in D],
              [(Q(9,2)*C[i]+3*Z[i])/(-3888) for i in range(3)],
              [Q(2*x,-3888) for x in Z]])
    assert F0*K0==want
    return dict(exceptional_prime_slopes=slopes,nonvanishing_determinant=str(det),initial_readout_residuals=9)

def rates(beta):
    # Unique positive saddle of the terminating denominator sum.
    P=lambda x:-180*x**3-720*x**2+6480*x+28800
    lo,hi=F(6),F(7)
    assert P(lo)>0>P(hi)
    for _ in range(220):
        mid=(lo+hi)/2
        if P(mid)>0:lo=mid
        else:hi=mid
    x=ball(lo).union(ball(hi));f=lambda z:z*z.log()
    alpha_f=f(x+12)+f(x+6)+f(x+5)-2*f(x+2)-2*f(x)-f(10-x)-f(x+1)-f(8-x)
    d=arb(GAMMA_CONSTANT).log()+sum((m*A*arb(A).log() for A,b,m in GAMMA),arb(0))
    alpha=alpha_f+d
    coeff=[F(1),-F(162060225797,305977695940),F(40100849267287098349209,374489401651004369934400),-F(25,7161587766945221818712445059146)]
    r=F('3.25999287587431764165969620794e-29');rl,rh=r-F('1e-58'),r+F('1e-58')
    poly=lambda z:sum(c*z**(3-i) for i,c in enumerate(coeff))
    assert poly(rl)<0<poly(rh)
    assert 3*rl**2+2*coeff[1]*rh+coeff[2]>0
    b,c,d0=coeff[1:]
    assert b*b*c*c-4*c*c*c-4*b*b*b*d0-27*d0*d0+18*b*c*d0<0
    E=(ball(-d0)/ball(rh)).log()/2-ball(rh).log()
    H=alpha-beta;delta=E/H-1
    assert beta>ball(F('-1.3335'))
    assert alpha<ball(F('63.13604'))
    assert E>ball(F('64.47615'))
    assert H<ball(F('64.46954'))
    assert delta>ball(F('0.0001'))
    return dict(content_rate_lower=str(beta),forward_height_rate=str(alpha_f),determinant_rate=str(d),raw_cofactor_rate=str(alpha),accepted_error_rate_lower=str(E),reduced_height_rate_upper=str(H),delta_lower=str(delta),advertised_delta='0.00010',saddle_interval=[str(lo),str(hi)])

def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--fresh',action='store_true')
    parser.add_argument('--output-dir',type=Path,default=Path(__file__).resolve().parent/'positive_certificate_run')
    args=parser.parse_args();args.output_dir.mkdir(parents=True,exist_ok=True)
    ctx.prec=256;aux=auxiliary_checks();T=20
    sha=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    checkpoint=args.output_dir/'checkpoint.json'
    saved=json.loads(checkpoint.read_text()) if checkpoint.exists() and not args.fresh else {}
    cache=saved.get('cells',{}) if saved.get('source_sha256')==sha else {}
    started=time.monotonic()
    def get(a,b,e,L):
        key=f'{e}:{L}:{a}:{b}'
        if key not in cache:
            cache[key]=cell(a,b,e,L)
            checkpoint.write_text(json.dumps(dict(source_sha256=sha,cells=cache),indent=2)+'\n')
        return cache[key]
    head=[];tail=[];finite=F(0)
    for e in (1,5):
        W=walls(e)
        for a,b in zip(W,W[1:]):
            for L in (0,1):
                dual=get(a,b,e,L)['common_bound'];forward=baseline(a,b,e,L)['combined_bound']
                v=max(dual,forward)
                rec=dict(mod6=e,lo=str(a),hi=str(b),lift=L,dual=dual,forward=forward,bound=v)
                if L==0:
                    head.append(rec)
                    if not a:assert v==0
                    else:finite+=v*(1/a-1/b)/2
                else:tail.append(rec)
        print(f'Base chambers for p mod 6 = {e} checked.',flush=True)
    for L in range(1,T):
        for r in tail:
            a,b=F(r['lo']),F(r['hi']);v=r['forward']
            if L==1:v=r['bound']
            elif r['dual']>r['forward']:
                v=max(v,get(a,b,r['mod6'],L)['common_bound'])
            finite+=v*(1/(L+a)-1/(L+b))/2
        print(f'Lift {L}/{T-1}: {len(cache)} exact chambers; {time.monotonic()-started:.1f}s',flush=True)
    beta=12*arb(2).log()+18*arb(3).log()+ball(finite)
    tail_integral=arb(0)
    for r in tail:
        a,b=F(r['lo']),F(r['hi'])
        tail_integral+=r['forward']*(ball(T+b).digamma()-ball(T+a).digamma())/2
    beta+=tail_integral
    result=dict(certificate_checks_passed=True,analytic_assumptions=['The endpoint/CMF readouts converge to C and zeta(2).','Their errors have exponential upper rate E as stated in PROOF.txt.'],
                source_sha256=sha,tail_start=T,exact_dual_chambers=len(cache),
                rational_finite_prime_density=str(finite),periodic_forward_tail_density=str(tail_integral),
                auxiliary_checks=aux,**rates(beta),seconds=time.monotonic()-started)
    (args.output_dir/'result.json').write_text(json.dumps(result,indent=2)+'\n')
    (args.output_dir/'base_chambers.json').write_text(json.dumps(dict(head=head,tail=tail),indent=2)+'\n')
    print(json.dumps(result,indent=2),flush=True)

if __name__=='__main__':main()
'''
certificate_path = OUT/'positive_delta_certificate.py'
certificate_path.write_text(arithmetic_source)
print('Arithmetic source SHA256:', hashlib.sha256(certificate_path.read_bytes()).hexdigest())


Arithmetic source SHA256: 949816687cdd5d836326a88be7dca030f1a0853174de395eab0d856557558932


In [14]:
certificate_dir=OUT/'certificate'
command=[sys.executable,'-u',str(certificate_path),'--output-dir',str(certificate_dir)]
if FRESH_ARITHMETIC:command.append('--fresh')
print('Starting exact arithmetic certificate; progress follows.',flush=True)
begun=time.monotonic()
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
transcript=[]
for line in process.stdout:
    transcript.append(line)
    # The final JSON is saved in full; show progress and the concise result here.
    if line.startswith('Base ') or line.startswith('Lift '):print(line.rstrip(),flush=True)
returncode=process.wait()
(OUT/'arithmetic_run.log').write_text(''.join(transcript))
assert returncode==0, ''.join(transcript[-30:])
certificate=json.loads((certificate_dir/'result.json').read_text())
assert certificate['certificate_checks_passed']
assert certificate['exact_dual_chambers']==1158
assert certificate['source_sha256']==hashlib.sha256(certificate_path.read_bytes()).hexdigest()
for key in ['exact_dual_chambers','content_rate_lower','raw_cofactor_rate',
            'accepted_error_rate_lower','reduced_height_rate_upper','delta_lower']:
    print(key,':',certificate[key])
print('Arithmetic mode:', 'fresh' if FRESH_ARITHMETIC else 'source-hashed checkpoint')
print('Certificate seconds:',round(time.monotonic()-begun,3))


Starting exact arithmetic certificate; progress follows.
Base chambers for p mod 6 = 1 checked.
Base chambers for p mod 6 = 5 checked.
Lift 1/19: 1158 exact chambers; 2.0s
Lift 2/19: 1158 exact chambers; 2.0s
Lift 3/19: 1158 exact chambers; 2.0s
Lift 4/19: 1158 exact chambers; 2.0s
Lift 5/19: 1158 exact chambers; 2.0s
Lift 6/19: 1158 exact chambers; 2.0s
Lift 7/19: 1158 exact chambers; 2.0s
Lift 8/19: 1158 exact chambers; 2.0s
Lift 9/19: 1158 exact chambers; 2.0s
Lift 10/19: 1158 exact chambers; 2.0s
Lift 11/19: 1158 exact chambers; 2.0s
Lift 12/19: 1158 exact chambers; 2.0s
Lift 13/19: 1158 exact chambers; 2.0s
Lift 14/19: 1158 exact chambers; 2.0s
Lift 15/19: 1158 exact chambers; 2.0s
Lift 16/19: 1158 exact chambers; 2.0s
Lift 17/19: 1158 exact chambers; 2.0s
Lift 18/19: 1158 exact chambers; 2.0s
Lift 19/19: 1158 exact chambers; 2.0s
exact_dual_chambers : 1158
content_rate_lower : [-1.333458772445433396272377612397984241835443307958351708976182808127960341 +/- 4.84e-73]
raw_cofactor_

## 8. Close the implication and check nonvanishing

The certificate's hard-coded characteristic coefficients must match the polynomial
derived from the actual CMF; this was checked in Section 6. Its root isolation and
final inequalities use rational arithmetic and Arb, independently of the decimal
`nroots` display. The certificate's legacy key `accepted_error_rate_lower` now
refers to the error bound derived above. Section 6A also identifies the limits;
both analytic prerequisites of that standalone verifier have been supplied.

For a nonzero fixed $(A,B)$, the error row
$3(A+B)X+2(A-B)Y-3\alpha_{A,B}D$ is nonzero. The next rational determinant
shows it cannot annihilate three consecutive sufficiently late state vectors.
It is a rational function of $n$, nonzero at $n=0$, so it has only finitely
many zeros or poles. Thus the error is nonzero infinitely often.

If the target were $a/b\in\mathbb Q$, every nonzero error would be at least
$1/(|b|q_N)$, contradicting $E>H$ where
$H=\alpha_{\rm raw}-\beta_{20}$. Hence the target is irrational. Then
$q_N\to\infty$, and
\[
\liminf\delta_N(A,B)\ge \frac{E}{H}-1>0.00010.
\]
The coefficient map $(A,B)\mapsto(A+B,\tfrac23(A-B))$ is invertible over
$\mathbb Q$; this proves the stated linear-independence implication.


In [15]:
import importlib.util
spec=importlib.util.spec_from_file_location('arithmetic_cert',certificate_path)
certmodule=importlib.util.module_from_spec(spec);spec.loader.exec_module(certmodule)
for N,key in [(0,'Q0'),(1,'Q1')]:
    native=Q.subs({n:N})
    stored=Matrix([[sp.Rational(z) for z in row] for row in certmodule.NONVANISHING[key]])
    assert native==stored
T0=Q.subs({n:0}).inv().T;T1=Q.subs({n:1}).inv().T
nonzero=Matrix.hstack(e0,T0*e0,T0*T1*e0).det()
assert nonzero==sp.Rational(certmodule.NONVANISHING['determinant_at_n0']) and nonzero!=0
ctx.prec=256
beta=arb(certificate['content_rate_lower'])
alpha_raw=arb(certificate['raw_cofactor_rate'])
E_lower=arb(certificate['accepted_error_rate_lower'])
height_upper=arb(certificate['reduced_height_rate_upper'])
delta_lower=arb(certificate['delta_lower'])
assert beta>arb('-1.3335') and alpha_raw<arb('63.13604')
assert E_lower>arb('64.47615') and height_upper<arb('64.46954')
assert delta_lower>arb('0.00010')
assert arb('64.47615')/arb('64.46954')-1>arb('0.00010')
print('Three-consecutive-column determinant:',nonzero)
print('Certified coarse comparison:',arb('64.47615')/arb('64.46954')-1)
print('Certified finer lower comparison:',delta_lower)
print('Conclusion: for every fixed nonzero integer (A,B), liminf delta > 0.00010,')
print('with both limit identifications and the error upper bound derived above.')
summary=dict(scope='analytic limits and error bound derived; arithmetic uses the written valuation lemmas',
    analytic_assumptions_remaining=[],
    analytic_certificate_sha256=hashlib.sha256(analytic_path.read_bytes()).hexdigest(),
    arithmetic_execution_mode='fresh' if FRESH_ARITHMETIC else 'source-hashed checkpoint',
    error_rate='derived from exact matrix series, scalar asymptotics and observability',
    error_exponent=str(sp.N(E,55)),delta_lower=certificate['delta_lower'],
    arithmetic_chambers=1158,versions=versions,depths=depths,
    native_dimension=4,restricted_dimension=3,forward_image_dimension=3,
    numerical_slopes=slopes,certificate_sha256=certificate['source_sha256'])
(OUT/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')


Three-consecutive-column determinant: -2091300253601094307522404926786910050752593888802048297/255217484616350899365532475817984000
Certified coarse comparison: [0.0001025290392951462039282427019023247257542088868634707181096685349391355980 +/- 6.46e-77]
Certified finer lower comparison: [0.000103328938306945387419974368047054100806454852732317739160357744 +/- 1.88e-67]
Conclusion: for every fixed nonzero integer (A,B), liminf delta > 0.00010,
with both limit identifications and the error upper bound derived above.
Out[15]: 2760


## Appendix A. Complete arithmetic argument

The text below is the arithmetic argument accompanying the standalone verifier.
It takes the limit and error rate as inputs; **Sections 6 and 6A of this notebook
prove both.** The normalization, packet-module lemmas,
small-prime estimates, forward tail and nonvanishing arguments are reproduced
in full for auditability. Exact calculation of the chambers checks the finite
algebraic component; the general valuation lemmas are mathematical arguments,
not claims established merely by testing a list of primes.


```text
POSITIVE DELTA FOR EVERY FIXED INTEGER MIXTURE: ARITHMETIC ARGUMENT
MG(5,2,5,5,1), 22 October 2026

This argument takes as input the limit and exponential error bound of
Section 1. Both are proved in ANALYTIC_PROOF.txt (Section 6A of the
notebook). Finite-depth delta measurements are not used in the arithmetic
calculation. The Python certificate verifies the finite chamber calculations
and numerical inequalities; the uniform interpretation of those calculations
is supplied by the lemmas below.

MAIN CONCLUSION

For every fixed (A,B) in Z^2 other than (0,0), put

  alpha(A,B) = A(C+2 Z/3)+B(C-2 Z/3),
  C=L(2,chi_{-3}),  Z=zeta(2).

Under the accepted analytic assumptions, the indicated CMF readouts satisfy

  liminf delta_N(A,B) > 0.00010.

In particular every alpha(A,B) is irrational, and 1,C,Z are linearly
independent over Q. The constant 0.00010 is a deliberately rounded lower
bound. It is not the much larger delta of the special minus readout.

1. THE SEQUENCE AND THE ANALYTIC ASSUMPTIONS

Use MG(5,2,5,5,1) at z=1, with

  start = (-1,-1,2,7/3,5/2; 0,0,0,1/3,1/2),
  direction = (-10,-8,12,6,5; 0,1,2,0,2).

Let G(n) be its standard four-dimensional trajectory matrix. Its last row
has zero entries in columns 0,1,2. Write Q(n)=G(n)[0:3,0:3],
W_N=Q(0)...Q(N-1), U_N=W_N^{-T}, and v_N=U_N e_0.
This is the existing restriction from dimension four to dimension three.
No additional reduction of the whole CMF is asserted.

The fixed rows are

  D=(243648,-498096,264600),
  X=(190344,-389162,206743),
  Y=(400824,-819330,435234).

Set r_C=(X v_N)/(D v_N), r_Z=(Y v_N)/(D v_N).
We assume r_C -> C, r_Z -> Z, and, for every epsilon>0,

  |r_C-C|+|r_Z-Z| <= exp(-(E-epsilon)N)

for all sufficiently large admissible N, where

  E = 64.476159691897769244147439665711... .

The same assumption then holds for every fixed linear mixture, up to a
constant depending on A,B. We do not assume any arithmetic denominator
bound. Denominators are always reduced scalar denominators.

2. THE DUAL ENDPOINT AND ITS TRANSPORT

Define the rational function

 F_n(s) = (s+1)_(10n+1) (s+1-n)_(9n+1)
          / [(2n-s)_(10n+2) (1/3-s)_(6n+2)
             (2n+1/2-s)_(3n+2)],

where (x)_m is the rising factorial. Its degree is -4. Its poles are
the integer, third-integer and half-integer progressions in the denominator.
For a pole rho=j+r, put v_rho=-Res_{s=rho} F_n(s), and set

 h_0(j)=sum_{l=1}^j l^{-2},
 h_r(j)=sum_{l=0}^j (l+r)^{-2}  (r=1/3,1/2).

The rows B_0,B_I,B_T,B_H, in columns k=0,1,2,3, are

 B_0,k = -sum_rho v_rho h_r(j) rho^k,
 B_I,k = sum_integer v_rho rho^k,
 B_T,k = sum_third v_rho rho^k,
 B_H,k = sum_half v_rho rho^k.

The residue theorem gives B_I+B_T+B_H=0 in columns 0,1,2.
Let Bhat_n be rows (B_0,B_T,B_H), columns (0,1,2). Then

                  Bhat_(n+1) = -Bhat_n Q(n).                 (2.1)

Here is an all-n justification, rather than an interpolation from examples.
The unnormalized Mellin--Barnes kernel of this Meijer-G field is

 I_n(s)= [prod_j Gamma(b_j-s) prod_{i=0,1} Gamma(1-a_i+s)]
         / prod_{i=2,3,4} Gamma(a_i-s)
       = (-1)^n pi^2 csc^2(pi s) F_n(s).                    (2.2)

The last equality follows twice from Gamma reflection. Gamma recurrence
is exactly the contiguous relation used by the Meijer-G constructor.
Take the Barnes contour Re(s)=-1/2, between the left and right poles;
it remains admissible along this walk. The moments through order three
converge. Small contours enclosing the finite third- and half-pole families
give B_T and B_H, up to fixed nonzero constants and the factor (-1)^n.
Zero residues at an endpoint permit the same contours to be used during
integer parameter shifts. Consequently those two rows obey (2.1).

For completeness the rational row can be extracted without assuming any
linear independence of C and Z. Close the Barnes contour to the right.
For k<=2 use F_n(s)s^k=-sum_rho v_rho rho^k/(s-rho). At an integer pole j,
the csc^2 expansion supplies the correction pi^2/3. Summing the other
integer residues gives H_j^(2)+Z; hence their net coefficient is
H_j^(2)-Z. At a noninteger pole j+r, reflection for the trigamma function
gives the net coefficient h_r(j)-psi_1(r). Thus, up to contour orientation,
the Barnes period row is

  B_0 + Z B_I + psi_1(1/3) B_T + psi_1(1/2) B_H
  = B_0 + (9 C/2+3 Z) B_T + 2 Z B_H.

This is an identity of explicitly evaluated periods, not an argument
equating coefficients of purportedly independent constants. Subtracting
the two already transported residue rows proves transport of B_0 as well.
The zero lower-left block of G gives (2.1) for the restriction.

Let w_n be the cross product of columns 1 and 2 of Bhat_n. Direct initial
evaluation, followed by (2.1), gives

  r_C = (4 w_1-6 w_2)/(18 w_0),    r_Z=w_2/(2 w_0),

and therefore

  r(A,B) = 2[(A+B)w_1-3B w_2]/(9w_0).                     (2.3)

Here the complete initial matrix, not just one initial vector, is

 Bhat_0=[[-789349/1680,-108221/140,-35787/35],
         [1647/35,3636/35,5328/35],
         [768/35,-1116/35,-2808/35]],  det(Bhat_0)=-1944/35.

If A_0=Bhat_0^(-T), its rows satisfy
3888 A_0[0,:]=D, 3888(4 A_0[1,:]-6 A_0[2,:])/18=X,
and 3888 A_0[2,:]/2=Y. These nine rational equalities identify the
fixed readouts for every transported vector.

At n=0 the cofactor vector is

  w_0 = (-121824/35, -205902/7, -400824/35).

For a rational vector w define its positive rational content by

  cont(w)=gcd(reduced numerators)/lcm(reduced denominators).

Equivalently v_p(cont(w))=min_i v_p(w_i). If u=w/cont(w) is primitive
integral, the actual denominator in (2.3) is

 q_n(A,B)=9|u_0| / gcd(9u_0,2[(A+B)u_1-3B u_2]) <=9|u_0|. (2.4)

The joint content gives a sufficient uniform bound. It is not being
asserted to be the exact pairwise gcd for every A,B.

3. THE FORWARD ENDPOINT: THE SECOND ARITHMETIC PRESENTATION

The forward field is 5F4 at z=1 with parameters

 a=(-1-10n,-1-8n,2+12n,7/3+6n,5/2+5n),
 b=(1+n,1+2n,4/3,3/2+2n).

Use the rational gamma quotient

 R_n(t)=(t+2n+1)_(10n+1) (t+4/3)_(6n+1)
        (t+2n+3/2)_(3n+1)
        /[(t-10n-1)_(10n+2) (t-8n-1)_(9n+2)].

It has degree -1, double poles at 0,...,8n+1 and single poles at
-n,...,-1 and 8n+2,...,10n+1. Let

 P_n(t)=(19440n^4+13812n^3+3219n^2+308n+11)/3
       -(4320n^3+4410n^2+1045n+69)t/6
       -(1080n^2+315n+7)t^2/6 +3t^3.

Write

 S_n(t)=R_n(t)P_n(t)=J_n(t)+sum_k[a_k/(t-k)^2+b_k/(t-k)],

where J_n is polynomial of degree two. The coefficient of 1/t at infinity
is exactly zero, so sum_k b_k=0. This can also be checked by the three
root-power differences saved in forward_infinity_identity.json.

For rational rho and integer m use the signed finite harmonic sums

 H_rho^(j)(m)=sum_{l=0}^{m-1}(rho+l)^(-j)   if m>=0,
             -sum_{l=m}^{-1}(rho+l)^(-j)   if m<0.

No zero denominator occurs in the instances below. Use

 (rho_I,m_I)=(1,2n), (rho_T,m_T)=(1/3,1),
 (rho_H,m_H)=(1/2,2n+1).

Define rational numbers

 q^f_n=sum_k a_k,
 p^f_r=sum_k[a_k H_rho_r^(2)(k+m_r)
             -b_k H_rho_r^(1)(k+m_r)]
        -sum_{j=0}^2 [t^j]J_n(t) (-1)^j zeta(-j,rho_r+m_r),

where zeta(-j,x)=-B_(j+1)(x)/(j+1) is a rational Bernoulli polynomial.
The forward rational vector is

             f_n=(q^f_n,p^f_T-p^f_I,p^f_H-p^f_I).

Its exact relation to the dual vector is

  f_n = (35/1944) [prod_{j=0}^{n-1} det Q(j)]^(-1) w_n.    (3.1)

To justify this identity uniformly, R_n(t) is exactly
prod_i Gamma(t+a_i)/(Gamma(t+1)prod_j Gamma(t+b_j)). Its double-pole
coefficient at t=0 is

 a_n,0=Gamma(12n+2)Gamma(6n+7/3)Gamma(5n+5/2)
       /[Gamma(2n+1)Gamma(4/3)Gamma(2n+3/2)
         Gamma(10n+2)Gamma(8n+2)Gamma(n+1)].

The sum of its double-pole moments is a_n,0 times the terminating 5F4
solution. Its simple-residue moments give the rational solution removed
by the existing restriction.

The sums of R_n(-rho_r-m) z^(-rho_r-m), starting at m=m_r, are the three
infinity Frobenius solutions for the positive upper parameters. The initial
terms vanish until t=-a_2,-a_3,-a_4, respectively, because of the numerator
root intervals. Integer parameter shifts only add/remove these zero terms.
All three series have the same logarithmic coefficient at z=1, since
R_n(t)=1/t+O(t^-2). Their differences are regular there. Partial fractions,
or Abel finite parts with z=exp(epsilon), evaluate these differences as

 (psi_1(rho_r)-psi_1(1)) times the double-pole row,
 plus a fixed digamma difference times the simple-residue row,
 minus the rational row (p^f_r-p^f_I).

The displayed Bernoulli term is precisely the finite part of the polynomial
piece. This explains why dropping that term or the starting-point shifts
would give the wrong numerator. Gamma recurrence transports all these
rows by the forward matrix M(n), with scalar a_(n+1),0/a_n,0.

The adjoint pairing is the exact identity

   M(n) K(n+1) Q(n)^T=c(n)K(n),

with K as in the included forward_dual_identity.json; P_n is its
first column read as a polynomial. Its rows are

 [(19440n^4+13812n^3+3219n^2+308n+11)/3,
  (6n+1)(720n^2+443n+46)/6, -5(216n^2+75n+7)/6],
 [-(4320n^3+4410n^2+1045n+69)/6,(360n^2+115n+8)/2,1],
 [-(1080n^2+315n+7)/6,-2,0], [3,0,0].

Gamma recurrence gives c(n)a_(n+1),0/a_n,0=1. At n=0 the three rational
forward rows before multiplication by K are

 [41/3,35/3,35/3,35/3],
 [1807/16,1339/16,8495/144,37819/1296],
 [2353/54,818/27,3649/216,311/432].

Multiplying by K(0) gives -1/3888 times, respectively,
D, (9/2)X+3Y, and 2Y. The certificate checks all nine entries. Their
transport, the cofactor transformation in (2.1), and the initial value
35/1944 prove (3.1). Small-depth direct equalities are additional regression
tests; they are not the justification for all n.

4. THE EXACT NORMALIZATION SCALAR

Put Delta_n=prod_{j=0}^{n-1} det Q(j). There is a rational function g(n)
such that

 Delta_n=C_0^n g(n)/g(0) prod_(A,beta,m) [(beta)_(An)]^m,
 C_0=2^8 3^30,

with triples

 (12,2/3,-1), (8,1,1), (5,1/3,-1), (8,1/3,1),
 (6,1,-1), (5,1,1), (4,1/6,1), (9,1/2,-1),
 (6,2/3,1), (5,2/3,1), (4,1/3,-1).

Here

 g(n)=3936600(8n+1)(9n+1)(15n+2)(20n+3)(24n+1)^2
      /[(2n+1)(3n+1)^2(6n+1)(8n+3)(9n+2)(10n+3)
        (12n+1)(15n+1)(15n+4)(18n+1)^2(24n+5)(30n+7)].

The one-step identity is verified symbolically in forward_scale_gamma.py.
The sum of A*m is zero. The absolute prime-valuation mass of g(n)/g(0)
is O(log n), so it does not affect the exponential content rate.

5. THE ELEMENTARY FORWARD PRIME BOUND, INCLUDING THE INFINITE TAIL

For p>3 and p^2>132n+30 put x=n/p, e=p mod 6. Group the roots of R_n
by their integer representative r modulo p, and put y=r/p. Its high roots
are in the following intervals, with lattice offsets as indicated:

 numerators: (-12x-y,-2x-y), offset 0;
             (-6x-y,-y), offset 2/3 if e=1, 1/3 if e=5;
             (-5x-y,-2x-y), offset 1/2;
 denominators: (-y,10x-y), (-x-y,8x-y), both offset 0.

Endpoints have the exact constant offsets -1, -4/3, -3/2 in the
numerators, and 0,1 in each denominator interval. The certificate includes
the boundary strips; it does not simply discard endpoint roots.

If a packet has h=#numerator roots-#denominator roots, double-pole
coefficients have valuation >=h+2 and simple coefficients >=h+1.
The harmonic sums cost at most two and one powers of p, respectively.
P_n, its derivatives, and the Bernoulli polynomial part are p-integral.
Consequently

                v_p(cont(f_n)) >= min(0,min_packets h).     (5.1)

Let this last bound be b_e(x). For beta=u/v in the gamma table, let
theta_e(beta) be the member of (0,1] congruent to u*e^(-1)/v modulo 1.
Away from a bounded neighborhood of the affine boundaries, Legendre
counting gives the determinant contribution

       d_e(x)=sum_(A,beta,m) m floor(Ax+1-theta_e(beta)).

By (3.1), a bound for the dual content is b_e(x)+d_e(x). This is the
forward bound in the common normalization used throughout the calculation.

For x=L+f, L>=1 and 0<f<1, this bound is EXACTLY periodic in L.
Indeed every high-root endpoint shifts by its integer slope times L;
the difference of total interval lengths has coefficient
(10+6+3)-(10+9)=0. The endpoint strips have the same labels. All intervals
are nonempty from L=1 onward. Also sum A*m=0 in d_e. Thus this is a
symbolic count argument for every L, not a test at a few integer lifts.
It supplies the entire infinite tail x>=20.

6. THE DUAL PACKET LEMMA USED IN THE FINITE RANGE

We use the dual presentation only for finitely many lifts 0<=L<20.
For the same large prime, write a pole s=r+pT with r in {0,...,p-1}.
The high numerator intervals for F_n are

  (-10x-y,-y), (-8x-y,x-y), both on the integer lattice;

and the high denominator intervals are

  (2x-y,12x-y), offset 0;
  (-y,6x-y), offset theta=1/3 (e=1) or 2/3 (e=5);
  (2x-y,5x-y), offset 1/2.

The constant endpoint offsets are (-1,-1),(-1,-1),(0,1),
(1/3,4/3),(1/2,3/2), respectively. These are used literally in the code.

For a packet define its monic high kernel

             f(T)=prod_numerator(T-a)/prod_denominator(T-b),
             h=deg(f),  R_b=Res_{T=b} f(T).

At a high pole b=j+theta, the p-divisible terms of the original harmonic
correction, multiplied by p^2, equal exactly H_theta^(2)(j+1), with
the integer case interpreted as H_j^(2). Denote this rational value by H(b).
The remaining terms are p-integral. Modulo p they are a common function
of r, minus a class-dependent constant: this follows by reducing a finite
reciprocal-square sum modulo p and using sum_{a=1}^{p-1} a^(-2)=0 mod p.
Subtract those class constants by an integral unipotent row operation.
This preserves the minimum valuation of the 2x2 minors.

Scale the resulting constant row by p^2, retain T,H and add the total
residue row I+T+H. Taylor expansion of all the unit factors in the packet
shows that its column vectors lie in the lattice generated by

 p^(h+1+j) V_j,  j=j_min,...,5,

 V_j=(-sum_b R_b b^j H(b),
       sum_third R_b b^j, sum_half R_b b^j, sum_all R_b b^j).

Here j_min=1 for r=0, otherwise 0. This is valid for BOTH selected
columns s and s^2; using 1 for the latter at r=0 merely weakens the bound.
The omitted Taylor terms are covered by arbitrary coordinate generators
of weight h+7. No convergence or floating rank test is involved.

The regular harmonic part is covered by the constant-row generator of
weight

  2+min(h+1+max(j_min,-h-1), h+2+j_min).                    (6.1)

To see this, its common leading value multiplies the total residue of
T^j f(T), which is zero while h+j<=-2. Its remaining part gains at
least one power of p. This gives exactly the two terms in the minimum.

Now impose the EXACT infinity-residue constraint I+T+H=0 on the selected
columns. The generator list always contains a weight-zero pivot with
total coordinate 1 or -1, and no negative-weight generator has a nonzero
total coordinate. The certificate verifies these properties. Subtracting
the pivot times the total coordinate therefore intersects the lattice
with that hyperplane by integral operations. This is the factor missed
by treating the packets independently without the infinity constraint.

For each two-row projection, sort the remaining generators by weight.
The least possible minor valuation is bounded below by the sum of the
weight of its first nonzero projected vector and the weight of its first
vector not rationally collinear with that vector. Subtract two if the
constant row is in the projection, undoing its p^2 scaling. Take the
minimum over the three projections. Rational collinearity is checked
exactly. A determinant becoming zero modulo p can only improve this
lower bound. This proves the dual bound calculated by cell().

For the finitely many lifts used, the high kernels and rational coefficients
belong to a finite list. Primes dividing their denominators can be excluded
as a fixed finite set; Section 8 bounds their total contribution by o(n).
For all other sufficiently large primes, the lattice argument above is
uniform in n and p throughout the indicated chamber.

7. EXACT CHAMBERS; WHY A FINITE CALCULATION IS SUFFICIENT

For e=1,5 define tau=2/3,1/3 respectively. Use the affine lines

 (-10,0),(0,0),(-8,0),(1,0),(2,0),(12,0),
 (0,tau),(6,tau),(2,1/2),(5,1/2),

where (u,a) means ux+a. The walls in [0,1] are all intersections of
differences of these lines with an integer. There are exactly 132 open
intervals for each e. All determinant-profile walls lie in the same list;
the verifier checks this inclusion.

For a fixed lift L, the high-root lists are constant between the resulting
affine y-cuts. Their boundary strips are enumerated with the exact rational
offsets stated above. Thus selecting an interior rational x and one point
in each y-stratum is an exhaustive chamber calculation. The interior
point uses denominator 1009 to avoid accidental coincidences of unrelated
scaled boundaries. It is not a sample of particular primes.

At every prime keep the MAXIMUM of the forward and dual lower bounds
in the common normalization (3.1). Do not add their gains.

The verifier computes:
  * both presentations for all 264 chambers at L=0;
  * both presentations for all 264 chambers at L=1;
  * the dual bound at L=2,...,19 for the 35 chambers where it improves
    the elementary forward bound at L=1;
  * the elementary periodic forward bound alone for every L>=20.

This is 1,158 exact dual chamber computations. There is no assumption that
the dual answer at L=1 repeats for higher L. In chambers not selected for
the extra finite computation we simply retain the valid forward bound.

Points within O(1/p) of an affine x-wall are harmless exceptions. For
p>sqrt(132n+30), there are O(sqrt(n)) such possible primes, since there
are finitely many walls per lift and O(sqrt(n)) lifts. Their elementary
valuation loss is O(log n/log p), so the total loss is o(n). Rational
coboundaries also cost only O(log n). Hence no endpoint range is silently
discarded at exponential scale.

8. SMALL PRIMES, AND THE EXCEPTIONAL PRIMES 2 AND 3

For 5<=p<=sqrt(132n+30), factorial/progression counting in the forward
residues and the determinant gamma product gives

               v_p(cont(w_n)) >= -O(log n/log p),

uniformly in p and in the pole index. The terms linear in n cancel:
there are 19n numerator and 19n denominator factors in the forward kernel,
and sum A*m=0 in the determinant product. Derivatives and the harmonic
corrections cost only a bounded number of additional logarithmic terms.
Summing the resulting losses gives O(sqrt(n) log n)=o(n).

At 2 and 3 the rational thirds and halves must be counted separately.
For every forward Laurent coefficient, and hence for every forward
readout, Legendre's formula gives the lower bounds

  v_2(cont(f_n)) >= -6n-O(log n),
  v_3(cont(f_n)) >= -9n-O(log n).

For example, at 2 the integer numerator block contributes 10n, the
third block 6n, the half block -3n, and the integer denominator blocks
-19n. At 3 these contributions are 5n,-6n,3n/2,-19n/2.
Simple-pole residues, derivatives, harmonic sums and P_n change these
counts by only O(log n). The polynomial part has fixed denominators.

The gamma product in Section 4 contributes 18n at 2 and 27n at 3, up to
O(log n). Therefore

  v_2(cont(w_n)) >=12n-O(log n),
  v_3(cont(w_n)) >=18n-O(log n).

The certificate independently computes those slopes from the gamma table.
These primes contribute 12 log 2+18 log 3 to the content rate.

9. PRIME DENSITY AND THE ARITHMETIC RESULT

The prime number theorem in progressions 1 and 5 modulo 6 gives density
one half for each class. A chamber a<n/p-L<b carrying lower valuation m
contributes

                  (m/2)[1/(L+a)-1/(L+b)].

Let m_e,L(a,b) denote the finite bounds actually computed, and f_e(a,b)
the elementary periodic forward bound. The certified lower content rate is

 beta_20 =12 log 2+18 log 3
          +1/2 sum_{e,cells,0<=L<20}
                m_e,L(a,b)[1/(L+a)-1/(L+b)]
          +1/2 sum_{e,cells} f_e(a,b)
                [psi(20+b)-psi(20+a)].                    (9.1)

The interval starting at a=0 has zero valuation at L=0 and contributes zero.
The digamma difference is the exact identity for the convergent sum over
L>=20. Boundedness of the periodic profile controls the remaining prime
tails uniformly, so PNT may be applied by truncation and then taking the
truncation to infinity.

All finite weights in (9.1) are exact rational numbers. The certificate
evaluates the logarithms and digamma values with Arb interval arithmetic.
It establishes

                beta_20 > -1.3335,
  liminf (log cont(w_n))/n >= beta_20.                     (9.2)

The unproved periodic extension of the observed dual profile would suggest
about -1.32853. That stronger number is NOT used. The proved forward tail
loses about 0.00493 and still leaves a positive margin.

10. RAW HEIGHT WITHOUT AN ADDITIONAL SPECTRAL ASSUMPTION

The double coefficients of R_n itself are positive for 0<=k<=8n+1.
The polynomial P_n costs only a polynomial factor. Stirling's formula
therefore bounds q^f_n by the maximum of the positive summand exponent

 phi(y)=(y+12)log(y+12)+(y+6)log(y+6)+(y+5)log(y+5)
        -2(y+2)log(y+2)-2y log y
        -(10-y)log(10-y)-(y+1)log(y+1)-(8-y)log(8-y),
           0<=y<=8.

The unique maximum is at the positive root y_* of

                 -180y^3-720y^2+6480y+28800=0.

Endpoint terms have the same uniform Stirling control. The leading value
of P_n(yn)/n^4 at the saddle is 6480-720y-180y^2<0. Positivity of the
double coefficients and the strict saddle therefore also show that q^f_n
is nonzero for all sufficiently large n. Thus

  limsup log|q^f_n|/n <= phi(y_*)
                      =32.90814001456892890464655... .

The determinant gamma product has rate

  sigma_det=log C_0+sum m A log A
           =30.22789934008875177435522... .

Equation (3.1) gives the unconditional raw cofactor height upper rate

  alpha_raw <=phi(y_*)+sigma_det
             =63.13603935465768067900177... <63.13604.

Combining this with (2.4) and (9.2), for EVERY fixed A,B,

             limsup log q_n(A,B)/n <64.46954.              (10.1)

This proof uses a terminating positive sum for the raw height; it does not
assume the raw-height conclusion from matrix asymptotics.

11. NONVANISHING AND THE FINAL IMPLICATION

Let T(n)=Q(n)^(-T). The rational function

  det[e_0,T(n)e_0,T(n)T(n+1)e_0]

has at n=0 the nonzero exact value

 -2091300253601094307522404926786910050752593888802048297
 /255217484616350899365532475817984000.

The certificate verifies this using the two exact matrices Q(0),Q(1).
Thus it has only finitely many zeros or poles. The three consecutive
state vectors are independent for all sufficiently large n. The fixed
row determinant det[D;X;Y]=-117573120 is also checked. For nonzero A,B,
the error row 3(A+B)X+2(A-B)Y-3 alpha(A,B) D is nonzero. It cannot
annihilate three consecutive sufficiently late states. Consequently the
approximation error is nonzero infinitely often, without presupposing
irrationality of its limit.

If alpha(A,B)=u/v were rational, each such nonzero error would be at least
1/(|v|q_n(A,B)). This contradicts the accepted exponential error bound and
(10.1), since E>64.47615>64.46954. Hence alpha(A,B) is irrational.
The denominators consequently tend to infinity. Finally

  liminf delta_n(A,B)
     >= E/(alpha_raw-beta_20)-1
     > 64.47615/64.46954-1
     > 0.00010.

The coefficient map (A,B) -> (A+B,(2/3)(A-B)) is invertible over Q.
Irrationality of every nonzero rational combination of C,Z is therefore
equivalent to linear independence of 1,C,Z.

12. REPRODUCTION

Run, from any directory with Python and python-flint installed,

  python positive_delta_certificate.py --fresh

It requires no saved approximants, target values, empirical prime profile,
or ramanujantools installation. It checkpoints exact chamber computations
and writes result.json in its output directory. The source hash binds a
checkpoint to this version of the verifier. The final JSON is written only
after all exact and interval-arithmetic assertions pass.

The optional symbolic_pairing_certificate.py independently reconstructs
the two CMF matrices using ramanujantools and verifies all twelve rational
pairing identities, the exact determinant gamma factorization, the zero
infinity residue, and the forward gamma shift. Its output is included as
symbolic_pairing_result.json. This symbolic check needs SymPy and
ramanujantools in addition to the arithmetic verifier's dependencies.

The code certifies the arithmetic calculation conditional on the mathematical
lemmas in this text, in the ordinary sense of a computer-assisted proof;
it is not a formal proof-assistant development. The analytic limit and error
inputs are proved separately in ANALYTIC_PROOF.txt.

As a separate regression check, the combined chamber bound was compared
with exact prime valuations at N=3928 and N=20480, excluding the stated
boundary/coboundary exceptions. All 71,593 eligible comparisons passed.
Those finite checks are recorded in the earlier regression run (not included in this bundle)
and are not used to justify the uniform inequalities.

Numerical uncertainty: no empirical delta enters the arithmetic certificate.
The certified delta is a conservative lower bound, not an
estimate of the optimal limiting delta.
```


## Appendix B. References

- RamanujanTools (commit pinned in `requirements.txt`) documents its Birkhoff–Trjitzinsky
  conventions in `asymptotics/growth_rate.py` and `asymptotics/reducer.py`.
- Background on the distinction between dominant modes and exceptional initial
  conditions: [Borcea, Friedland and Shapiro, *Parametric Poincaré–Perron theorem
  with applications*](https://staff.math.su.se/shapiro/Articles/Poincare-Perron.pdf),
  especially Theorems 3–4. A spectral gap alone does not identify a target constant.
- For summable perturbations and asymptotic fundamental systems, see Section 7 of
  [Baumann and Hanschke, *Computation of solutions to linear difference and differential
  equations with a prescribed asymptotic behavior*](https://link.springer.com/article/10.1186/s13662-021-03333-9).
